## Methylation Stats

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib.patches as mpatches
from matplotlib.cm import get_cmap
import matplotlib
import pybedtools
from pybedtools import BedTool
from tqdm import tqdm 
from pG4utils.utils import parse_fasta
from pG4utils.vcftools import TrinucleotideModel
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import patches as mpatches
from statannotations.Annotator import Annotator
from scipy.stats import wilcoxon

target = Path().cwd().joinpath("g4_t2t_revisions_data"); target.mkdir(exist_ok=True)
dataset_path = Path("/scratch/10904/nikolchanchan/data")

# # pG4s # 
G4HUNTER = dataset_path / "pG4s_extractions" / "g4hunter" / "chm13v2_g4hunter.txt.gz"
QUADPARSER = dataset_path / "pG4s_extractions" / "quadparser" / "chm13v2_regex_motifs.txt"
EG4 = Path("/work/10904/nikolchanchan/vista/g4_revisions/g4_t2t_revisions/notebooks") / "eG4.txt"

In [ ]:
datasets = {
            "G4Hunter": pd.read_table(G4HUNTER),
            "Quadparser": pd.read_table(QUADPARSER),
            "eG4": pd.read_table(EG4).rename(columns={"Chr": "Chromosome", 
                                                      "Start": "Start", 
                                                      "End": "End"})
                                                
    }
datasets["eG4"]

In [ ]:
from tqdm import tqdm 
from pG4utils.utils import parse_fasta

FASTA = dataset_path / "fasta" / "hs1.fa.gz"
dataset_path = Path("/scratch/10904/nikolchanchan/data")
FASTA = dataset_path / "hs1.fa.gz"
assert FASTA.is_file()

In [ ]:
target     = Path("/scratch/10904/nikolchanchan/G4_T2T/methylation_analysis/data")
target.mkdir(exist_ok=True, parents=True)

target_fig = Path("/scratch/10904/nikolchanchan/G4_T2T/methylation_analysis/figures")
target_fig.mkdir(exist_ok=True, parents=True)

In [ ]:
from tqdm import tqdm 
from pG4utils.utils import parse_fasta

FASTA = dataset_path / "fasta" / "hs1.fa.gz"
FASTA = dataset_path / "hs1.fa.gz"
assert FASTA.is_file()
# # # #
seq_sizes = pd.read_table(dataset_path / "genome.txt", 
                          header=None)
seq_sizes = dict(zip(seq_sizes[0], seq_sizes[1]))
seq_sizes


In [ ]:
# Load methylation
import os

def extract_methylation(meth_prob: float) -> str:
    if meth_prob < 0.2:
        return "Hypomethylated"
    if meth_prob < 0.8:
        return "Methylated"
    return "Hypermethylated"

methylation_HG002_df = pd.read_table(Path(os.getenv("SCRATCH")).joinpath("g4_t2t_revisions_data").joinpath("chm13v2.0_hg002_CpG_ont_guppy6.1.2.bedgraph.collapsed.gz"))
methylation_CHM13v2_df = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/chm13v2.0_CHM13_CpG_ont_guppy3.6.0_nanopolish0.13.2.bedgraph",
                                       header=None,
                                       names=["Chromosome", "Start", "End", "methylation_level"])
methylation_CHM13v2_df.head()

In [ ]:
def fetch_stats(df):
    if "sequence" not in df:
        df.loc[:, "sequence"] = df.apply(lambda row: chrom_sequences[row["seqID"]][row["start"]: row["end"]], axis=1)
    df.loc[:, "sequence"] = df["sequence"].str.lower()
    df.loc[:, "length"] = df["end"] - df["start"]
    df.loc[:, "gc_content"] = df["sequence"].str.count("[gc]")
    df.loc[:, "cpg"] = df["sequence"].apply(lambda seq: sum(int(seq[i:i+2] == "cg") for i in range(len(seq)-1)))
    df.loc[:, "gpc"] = df["sequence"].apply(lambda seq: sum(int(seq[i:i+2] == "gc") for i in range(len(seq)-1)))
    return df

In [ ]:
chrom_sequences = dict()
for seqID, seq in tqdm(parse_fasta(FASTA), total=25):
    chrom_sequences[seqID] = seq.upper()

In [ ]:
import pyranges as pr
import polars as pl

# auto-detect the methylation value column (the non-coordinate column)
METH_COL = "methylation_level"
remap = lambda df: df.rename(columns={"seqID": "Chromosome", 
                                      "start": "Start", 
                                      "end": "End"})
meth_HG002_pr = pr.PyRanges(remap(methylation_HG002_df))
meth_CHM13_pr = pr.PyRanges(remap(methylation_CHM13v2_df))

def _intersect_meth(motif_df, group_cols, meth_pr):
    if isinstance(motif_df, pl.DataFrame):
        motif_pd = motif_df.rename({"seqID": "Chromosome", 
                                    "Chr": "Chromosome", 
                                    "start": "Start", 
                                    "end": "End"}).to_pandas()
    else:
        motif_pd = motif_df.rename(columns={"seqID": "Chromosome", "Chr": "Chromosome", "start": "Start", "end": "End"})

    joined = pr.PyRanges(motif_pd).join(meth_pr, suffix="_meth")
    df = pl.from_pandas(joined.df)
    meth_col = f"{METH_COL}_meth" if f"{METH_COL}_meth" in df.columns else METH_COL
    return (
        df
        .filter(
            (pl.col("Start_meth") >= pl.col("Start")) &
            (pl.col("End_meth")   <= pl.col("End"))
        )
        .rename({"Chromosome": "seqID", "Start": "start", "End": "end"})
        .group_by(group_cols)
        .agg([
            pl.col(meth_col).mean().alias("avg_methylation"),
            pl.col(meth_col).count().alias("meth_count"),
        ])
        .with_columns(
            pl.col("avg_methylation")
              .map_elements(extract_methylation, return_dtype=pl.String)
              .alias("methylation_level")
        )
    )

METH_COL   = "methylation_level"
GROUP_COLS = ["seqID", "start", "end"]
remap = lambda df: df.rename(columns={"seqID": 
                                      "Chromosome", 
                                      "Chr": "Chromosome",
                                      "start": "Start", 
                                      "end": "End"})
meth_HG002_pr  = pr.PyRanges(remap(methylation_HG002_df))
meth_CHM13_pr  = pr.PyRanges(remap(methylation_CHM13v2_df))
samples = [
    ("HG002", meth_HG002_pr),
    ("CHM13", meth_CHM13_pr),
]

meth_annotated = {}
for sample_name, meth_pr in samples:
    for ds_name, ds_df in datasets.items():
        out = target / f"{ds_name}_motif_methylation_annot_{sample_name}.tsv.gz"
        result = _intersect_meth(ds_df, GROUP_COLS, meth_pr)
        result.write_csv(str(out), 
                         separator="\t", 
                         compression="gzip", 
                         include_header=True)
        meth_annotated[(ds_name, sample_name)] = result
        print(f"  {out.name}  ({len(result):,} rows)", flush=True)

In [ ]:
# print(meth_annotated["eG4", "HG002"].shape)
# annotated_eG4 = (
#                 meth_annotated["eG4", "HG002"]  
#                 .drop_duplicates()
#                 .merge(datasets["eG4"], 
#                                         on=["seqID", "start", "end"])
# )
# annotated_eG4

In [ ]:
meth_annotated = {}
for sample_name in ["HG002", "CHM13"]:
    for ds_name, ds_df in datasets.items():
        out = target / f"{ds_name}_motif_methylation_annot_{sample_name}.tsv.gz"
        result = pd.read_table(str(out))
        meth_annotated[(ds_name, sample_name)] = result
        print(f"  {out.name}  ({len(result):,} rows)", flush=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from pathlib import Path

records = []
for (ds_name, sample_name), df in meth_annotated.items():
    total = len(df)
    if isinstance(df, pd.DataFrame):
        df = pl.from_pandas(df)
    counts = df.group_by("methylation_level").agg(pl.len().alias("n"))
    for row in counts.iter_rows(named=True):
        records.append({
            "algorithm": ds_name,
            "sample":    sample_name,
            "category":  row["methylation_level"],
            "n":         row["n"],
            "pct":       row["n"] / total * 100,
        })

agg = pl.DataFrame(records)

algorithms = sorted(agg["algorithm"].unique().to_list())
categories = ["Hypomethylated", "Methylated", "Hypermethylated"]
palette    = {"HG002": "#4C72B0", "CHM13": "lightblue"}
x          = np.arange(len(categories)) * 1.9
width      = 0.5
offsets = {"HG002": -width / 2 - 0.04, "CHM13": width / 2 + 0.04}

fig, axes = plt.subplots(1, len(algorithms), figsize=(14, 5), sharey=True)
fig.subplots_adjust(hspace=0.00, wspace=0)
if len(algorithms) == 1:
    axes = [axes]

for col, algo in enumerate(algorithms):
    ax = axes[col]

    for j, sample in enumerate(["HG002", "CHM13"]):
        vals, pcts = [], []
        for cat in categories:
            sub = agg.filter(
                (pl.col("algorithm") == algo) &
                (pl.col("sample")    == sample) &
                (pl.col("category")  == cat)
            )
            vals.append(float(sub["n"][0])   if len(sub) else 0.0)
            pcts.append(float(sub["pct"][0]) if len(sub) else 0.0)

        bars = ax.bar(
            x + offsets[sample], pcts, width,
            color=palette[sample],
            alpha=0.85,
            edgecolor="black",
            linewidth=1.4,
            label=sample,
        )
        if sample == "CHM13":
            for bar in bars:
                bar.set_linestyle("--")

        pmax = max(pcts) if max(pcts) > 0 else 1
        for idx, (bar, n, pct) in enumerate(zip(bars, vals, pcts)):
            if j == 0:
                ax.text(
                    bar.get_x() - 0.15 + bar.get_width() / 2,
                    bar.get_height() + pmax * 0.012,
                    f"{int(n):,}\n{pct:.1f}%",
                    ha="center", va="bottom", fontsize=11,
                )
            else:
                ax.text(
                    bar.get_x() + 0.2 + bar.get_width() / 2,
                    bar.get_height() + pmax * 0.012,
                    f"{int(n):,}\n{pct:.1f}%",
                     ha="center", va="bottom", fontsize=11,
                )

    ax.set_ylim(0, ax.get_ylim()[1] * 1.2)
    ax.set_xlim(x[0] - 1.0, x[-1] + 1.0)
    ax.set_xticks(x)
    ax.set_axisbelow(True)
    ax.set_xticklabels(categories, fontsize=16, rotation=23)
    ax.set_xlabel("", fontsize=13)
    if col == 0:
        ax.set_ylabel("G4 Occurrences (%)", fontsize=20)
    else:
        ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=15)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.grid(axis="y", lw=0.4, alpha=0.6)
    ax.set_title(
        algo, fontsize=19, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#D3D3D3", edgecolor="gray", linewidth=1.2),
    )
    if col == 2:
        ax.legend(fontsize=14)
fig.subplots_adjust(hspace=0.00, wspace=0)
fig.tight_layout()
fig.savefig(str(target_fig / f"methylation_category_barplots.png"), dpi=300, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pybedtools import BedTool
import pandas as pd

METH_COL = "methylation_level"
FLANK = 100

def cpgs_internal_level(motif_df, meth_df, flank=100):
    """
    Returns a pl.DataFrame with columns:
        seqID, start, end, internal_cpgs, flanking_cpgs
    """
    if isinstance(motif_df, pl.DataFrame):
        motif_pd = motif_df.to_pandas()
    else:
        motif_pd = motif_df.copy()

    if isinstance(meth_df, pl.DataFrame):
        meth_pd = meth_df.to_pandas()
    else:
        meth_pd = meth_df.copy()
    motif_bed = BedTool.from_dataframe(motif_pd)
    meth_bed  = BedTool.from_dataframe(meth_pd)
    
    internal = (
        meth_bed
        .intersect(motif_bed, u=True, f=1.0)
        .to_dataframe(names=["seqID", "start", "end", METH_COL])
    )
    internal.loc[:, "level"] = internal[METH_COL].apply(extract_methylation)
    internal_counts = (
        pl.from_pandas(internal)
        .group_by(["level"])
        .agg(pl.len().alias("internal_cpgs"))
    )

    return internal_counts

counts_level = {}
meth_datasets = {
    "HG002": methylation_HG002_df,
    "CHM13": methylation_CHM13v2_df,
}
records_cpg = []
for tissue, df in meth_datasets.items():
    for label, motif_df in tqdm(datasets.items()):
        records_cpg.append(cpgs_internal_level(motif_df, df, flank=FLANK)
                           .with_columns(
                                 pl.lit(label).alias("algorithm"),
                                 pl.lit(tissue).alias("sample"),
                           )
        )

records_cpg = pl.concat(records_cpg)
records_cpg

In [ ]:
records_cpg_total = records_cpg.group_by(["algorithm", "sample"]).agg(total_cpg=pl.col("internal_cpgs").sum())
records_cpg = records_cpg.join(records_cpg_total, on=["algorithm", "sample"])
records_cpg = records_cpg.with_columns(
    perc=pl.col("internal_cpgs") / pl.col("total_cpg") * 100
)
records_cpg.write_csv(str(target / "cpg_counts_internal_levels.tsv.gz"), 
                separator="\t",
                compression="gzip",
                include_header=True)
records_cpg

In [ ]:
algorithms = sorted(records_cpg["algorithm"].unique().to_list())
categories = ["Hypomethylated", "Methylated", "Hypermethylated"]
palette    = {"HG002": "#4C72B0", "CHM13": "lightblue"}
x          = np.arange(len(categories)) * 1.9
width      = 0.5
offsets = {"HG002": -width / 2 - 0.04, "CHM13": width / 2 + 0.04}

fig, axes = plt.subplots(1, len(algorithms), figsize=(14, 5), sharey=True)
fig.subplots_adjust(hspace=0.00, wspace=0)
if len(algorithms) == 1:
    axes = [axes]

for col, algo in enumerate(algorithms):
    ax = axes[col]

    for j, sample in enumerate(["HG002", "CHM13"]):
        vals, pcts = [], []
        for cat in categories:
            sub = records_cpg.filter(
                (pl.col("algorithm") == algo) &
                (pl.col("sample")    == sample) &
                (pl.col("level")  == cat)
            )
            vals.append(float(sub["internal_cpgs"][0])   if len(sub) else 0.0)
            pcts.append(float(sub["perc"][0]) if len(sub) else 0.0)

        bars = ax.bar(
            x + offsets[sample], vals, width,
            color=palette[sample],
            alpha=0.85,
            edgecolor="black",
            linewidth=1.4,
            label=sample,
        )
        if sample == "CHM13":
            for bar in bars:
                bar.set_linestyle("--")

        pmax = max(pcts) if max(pcts) > 0 else 1
        for idx, (bar, n, pct) in enumerate(zip(bars, vals, pcts)):
            if j == 0:
                ax.text(
                    bar.get_x() - 0.1 + bar.get_width() / 2,
                    bar.get_height() + pmax * 0.012,
                    f"{pct:.1f}%",
                    ha="center", va="bottom", fontsize=11,
                )
            else:
                ax.text(
                    bar.get_x() + 0.1 + bar.get_width() / 2,
                    bar.get_height() + pmax * 0.012,
                    f"{pct:.1f}%",
                     ha="center", va="bottom", fontsize=11,
                )

    ax.set_ylim(0, ax.get_ylim()[1] * 1.2)
    ax.set_xlim(x[0] - 1.0, x[-1] + 1.0)
    ax.set_xticks(x)
    ax.set_axisbelow(True)
    ax.set_xticklabels(categories, fontsize=16, rotation=23)
    ax.set_xlabel("", fontsize=13)
    if col == 0:
        ax.set_ylabel("Total methylated sites", fontsize=20)
    else:
        ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=15)
    # ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.grid(axis="y", lw=0.4, alpha=0.6)
    ax.set_title(
        algo, fontsize=19, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#D3D3D3", edgecolor="gray", linewidth=1.2),
    )
    if col == 2:
        ax.legend(fontsize=14)
fig.subplots_adjust(hspace=0.00, wspace=0)
fig.tight_layout()
fig.savefig(str(target_fig / f"cpg_methylation_category_barplots.png"), 
            dpi=300, 
            bbox_inches="tight", 
            transparent=True)
plt.show()

In [ ]:
def count_cpgs_internal_flanking_seq(motif_df, chrom_sequences, flank=100):
    if isinstance(motif_df, pl.DataFrame):
        motif_pd = motif_df.to_pandas()
    else:
        motif_pd = motif_df.copy()

    internal_cpgs = []
    flanking_cpgs = []
    
    for _, row in motif_pd.iterrows():
        chrom = row["seqID"]
        start = int(row["start"])
        end   = int(row["end"])
        seq   = chrom_sequences[chrom]

        internal_cpgs.append(seq[start:end].count("CG"))

        up_seq = seq[max(0, start - flank):start]
        dn_seq = seq[end:end + flank]
        flanking_cpgs.append(up_seq.count("CG") + dn_seq.count("CG"))

    return (
        pl.from_pandas(motif_pd[["seqID", "start", "end"]])
        .with_columns([
            pl.Series("internal_cpgs",  internal_cpgs,  dtype=pl.Int32),
            pl.Series("flanking_cpgs",  flanking_cpgs,  dtype=pl.Int32),
        ])
    )

counts_seq = {}
for label, motif_df in tqdm(datasets.items()):
    counts_seq[label] = count_cpgs_internal_flanking_seq(motif_df, chrom_sequences, flank=FLANK)

In [ ]:
meth_annotated_further = dict()
for sample in ["HG002", "CHM13"]:
    for method in ["G4Hunter", "Quadparser", "eG4"]: 
        meth_annotated_further[method, sample] = meth_annotated[(method, sample)].merge(
            counts_seq[method].to_pandas(),
            on=["seqID", "start", "end"],   
            how="left"
        )
meth_annotated_further[method, sample]

In [ ]:
METH_COL = "methylation_level"
FLANK = 100
INT_CAP   = 10
FLANK_CAP = 100
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

row_cfg = [
    ("internal_cpgs",  INT_CAP,   "CpGs within G4",           1,  [str(i) for i in range(INT_CAP)] + [f"{INT_CAP}+"]),
    ("flanking_cpgs",  FLANK_CAP, f"Flanking CpGs (±{FLANK} bp)", 10, None),
]

for col_idx, (method, df) in enumerate(datasets.items()):
    df    = counts_seq[method]

    for row_idx, (cpg_col, cap, xlabel, tick_step, fixed_labels) in enumerate(row_cfg):
        ax   = axes[row_idx, col_idx]
        vals = np.clip(df[cpg_col].to_numpy(), 0, cap).astype(int)
        bar_counts = np.bincount(vals, minlength=cap + 1)

        ax.bar(np.arange(cap + 1), bar_counts,
               color="blue", 
               edgecolor="white", linewidth=1.0, width=0.85)

        if fixed_labels is not None:
            ax.set_xticks(np.arange(cap + 1))
            ax.set_xticklabels(fixed_labels, fontsize=13)
        else:
            ticks  = list(range(0, cap, tick_step)) + [cap]
            tlabels = [str(t) for t in range(0, cap, tick_step)] + [f"{cap}+"]
            ax.set_xticks(ticks)
            ax.set_xticklabels(tlabels, fontsize=13)

        ax.tick_params(axis="y", labelsize=13)
        ax.set_xlabel(xlabel, fontsize=16)
        ax.set_ylabel("Number of G4s", fontsize=16)
        ax.grid(axis="y", lw=0.4, alpha=0.6)
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        # rounded pill-box column title on top row only
        if row_idx == 0:
            ax.set_title(
                method, 
                fontsize=18, 
                fontweight="bold", 
                pad=12,
                bbox=dict(boxstyle="round,pad=0.45", facecolor="gray",
                          edgecolor="none", alpha=0.22),
            )

fig.tight_layout(h_pad=3.5, w_pad=2.5)
fig.savefig(f"{target_fig}/cpg_internal_flanking_distribution.png",
            dpi=300,
            bbox_inches="tight", 
            transparent=True)
plt.show()

## Analysis

In [ ]:
from scipy.stats import wilcoxon
import pandas as pd
import polars as pl
import os
from pathlib import Path 

scratch     = os.getenv("SCRATCH")
data_target = Path("/scratch/10904/nikolchanchan/g4_t2t_revisions_data")
matched = {}
G4_DATASETS = ["G4Hunter", "Quadparser"]
SAMPLES = ["HG002", "CHM13"]
eg4_median = {
    sample_name: (
        meth_annotated[("eG4", sample_name)]["avg_methylation"]
        .median()
    )
    for sample_name in SAMPLES
}
eg4_median

In [ ]:
def add_rank(meth_df, reptime_df):
    meth_pd = meth_df.to_pandas() if isinstance(meth_df, pl.DataFrame) else meth_df
    return meth_pd.merge(
        reptime_df[["seqID", "start", "end", "rank"]],
        on=["seqID", "start", "end"],
        how="inner"
    )


In [ ]:
def p_to_star(p):
    if p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

In [ ]:
chrom_sequences = dict()
for seqID, seq in tqdm(parse_fasta(FASTA), total=25):
    chrom_sequences[seqID] = seq.upper()

## Replication Time Deciles

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import patches as mpatches
from tqdm import tqdm
import os

rep_df = pd.read_csv(
    f"{os.getenv('SCRATCH')}/g4_t2t_revisions_data/bg02es_replitime.deciles.hs1.bed",
    header=None,
    sep="\t",
    names=["seqID", "start", "end", "decile", "rank"]
).sort_values(["seqID", "start"]).reset_index(drop=True)
rep_df["rank"] = rep_df["rank"].map({i: 11 - i for i in range(1, 11)})

def merge_overlapping(df: pd.DataFrame, col: str) -> pd.DataFrame:
    df_collection = []
    for c in tqdm(df[col].unique()):
        df_temp = pd.read_csv(
            BedTool.from_dataframe(df[df[col] == c]).sort().merge(c="3", o="count").sort().fn,
            header=None,
            sep="\t",
            names=["seqID", "start", "end", "counts"]
        )
        df_temp[col] = c
        df_collection.append(df_temp)
    return pd.concat(df_collection, ignore_index=True)

print(rep_df.shape)
rep_df_merged = merge_overlapping(rep_df, col="rank")
rep_time = BedTool.from_dataframe(rep_df_merged).sort()
rep_df_merged

In [ ]:
import pyranges as pr

rep_pr = pr.PyRanges(
    rep_df.rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
).merge(by="rank")

reptime_dfs = {}
for ds_name, ds_df in datasets.items():
    if isinstance(ds_df, pl.DataFrame):
        ds_pd = ds_df.rename({"seqID": "Chromosome", "start": "Start", "end": "End"}).to_pandas()
    else:
        ds_pd = ds_df.rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})

    joined = pr.PyRanges(ds_pd).join(rep_pr, suffix="_rep")
    reptime_dfs[ds_name] = (
        joined.df
        .loc[lambda d: (d["Start"] >= d["Start_rep"]) & (d["End"] <= d["End_rep"])]
        .drop(columns=["Start_rep", "End_rep"])
        .rename(columns={"Chromosome": "seqID", "Start": "start", "End": "end"})
        .reset_index(drop=True)
    )
    print(f"  {ds_name}: {len(reptime_dfs[ds_name]):,} intervals", flush=True)

In [ ]:
import numpy as np
def get_base_score(line: str) -> tuple[str, list[int]]:
    item, score_list = 0, []
    # calcule le item de chaque base et la stock dans score_list
    while (item < len(line)):
        if (item < len(line) and (line[item]=="G" or line[item]=="g")):
            score_list.append(1)
            if(item+1< len(line) and (line[item+1]=="G" or line[item+1]=="g")):
                score_list[item]=2
                score_list.append(2)
                if (item+2< len(line) and (line[item+2]=="G" or line[item+2]=="g")):
                    score_list[item+1]=3
                    score_list[item]=3
                    score_list.append(3)
                    if (item+3< len(line) and (line[item+3]=="G" or line[item+3]=="g")):
                        score_list[item]=4
                        score_list[item+1]=4
                        score_list[item+2]=4
                        score_list.append(4)
                        item=item+1
                    item=item+1
                item=item+1
            item=item+1
            while(item < len(line) and (line[item]=="G" or line[item]=="g")):
                    score_list.append(4)
                    item=item+1

        elif (item < len(line) and line[item]!="G" and line[item]!="g" and line[item]!= "C" and line[item]!="c" ):
                    score_list.append(0)
                    item=item+1
            
        elif(item < len(line) and (line[item]=="C" or line[item]=="c")):
            score_list.append(-1)
            if(item+1< len(line) and (line[item+1]=="C" or line[item+1]=="c" )):
                score_list[item]=-2
                score_list.append(-2)
                if (item+2< len(line) and (line[item+2]=="C" or line[item+2]=="c" )):
                    score_list[item+1]=-3
                    score_list[item]=-3
                    score_list.append(-3)
                    if (item+3< len(line) and (line[item+3]=="C" or line[item+3]=="c"  )):
                        score_list[item]=-4
                        score_list[item+1]=-4
                        score_list[item+2]=-4
                        score_list.append(-4)
                        item=item+1
                    item=item+1   
                item=item+1
            item=item+1
            while(item < len(line) and (line[item]=="C" or line[item]=="c")):
                score_list.append(-4)
                item=item+1
        else:
                item=item+1 
    # return line, score_list
    return np.mean(score_list)

In [ ]:
eg4_df = datasets["eG4"]
eg4_df.loc[:, "sequence"] = eg4_df.apply(lambda row: chrom_sequences[row["seqID"]][row["start"]:row["end"]], axis=1)
eg4_df.loc[:, "score"] = eg4_df["sequence"].apply(get_base_score)
eg4_df

In [ ]:
g4_df = pd.read_table(G4HUNTER)
g4_df["sequence"] = g4_df.apply(lambda row: chrom_sequences[row["seqID"]][row["start"]:row["end"]], axis=1)
g4_df["new_score"] = g4_df["sequence"].apply(get_base_score)
g4_df

In [ ]:
cov_fields = ["total_hits", "total_bases", "all_bases", "coverage"]
g4_bed = BedTool.from_dataframe(g4_df[["seqID", "start", "end"]]).sort()
eg4_bed = BedTool.from_dataframe(eg4_df[["seqID", "start", "end"]]).sort()
g4_with_eg4_df = pd.read_table(
    g4_bed.coverage(eg4_bed).fn,
    header=None,
    names=["seqID", "start", "end"] + cov_fields
)
THRESHOLD = 0.5
g4_with_eg4_df.loc[:, "is_eg4"] = (g4_with_eg4_df["coverage"] >= THRESHOLD).astype(int)
g4_with_eg4_df

In [ ]:
g4_with_eg4_df["is_eg4"].value_counts()

In [ ]:
regex_df = datasets["Quadparser"]
regex_df

In [ ]:
from scipy.stats import mannwhitneyu
import pyranges as pr
import pandas as pd
import numpy as np

FIELDS = ["seqID", "start", "end"]

def _to_pr(df):
    d = df[FIELDS].rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"}).copy()
    d["_g4_id"] = range(len(d))
    return pr.PyRanges(d)

g4_prs = {
    "G4Hunter":   _to_pr(g4_df),
    "Quadparser": _to_pr(regex_df),
    "eG4":        _to_pr(eg4_df),
}

RANKS = sorted(rep_df_merged["rank"].unique())
rep_prs = {
    rank: pr.PyRanges(
        rep_df_merged[rep_df_merged["rank"] == rank]
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
        [["Chromosome", "Start", "End"]]
    )
    for rank in RANKS
}

raw_meth = {"HG002": methylation_HG002_df, "CHM13": methylation_CHM13v2_df}

records = []
for sample_name in SAMPLES:
    meth_df = (
        raw_meth[sample_name]
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
        [["Chromosome", "Start", "End", "methylation_level"]]
        .dropna(subset=["methylation_level"])
    )
    full_cpg_pr = pr.PyRanges(meth_df)

    for rank in RANKS:
        cpg_pr = full_cpg_pr.intersect(rep_prs[rank])

        for database, g4_pr in g4_prs.items():
            joined = g4_pr.join(cpg_pr, how="left")
            locus_avg = (
                joined.df
                .dropna(subset=["methylation_level"])
                .query("methylation_level != -1")
                .groupby("_g4_id")["methylation_level"]
                .mean()
                .reset_index()
                .rename(columns={"methylation_level": "avg_meth"})
            )
            locus_avg["sample"]   = sample_name
            locus_avg["rank"]     = rank
            locus_avg["database"] = database
            records.append(locus_avg)

locus_df = pd.concat(records, ignore_index=True)
locus_df

In [ ]:
def map_meth(level: float) -> str:
    if level < 0.2:
        return "Hypomethylated"
    elif level < 0.8:
        return "Methylated"
    else:
        return "Hypermethylated"

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

rows = []
for sample in SAMPLES:
    for db in ["G4Hunter", "Quadparser", "eG4"]:
        sub = locus_df[
            (locus_df["sample"] == sample) &
            (locus_df["database"] == db)
        ]
        # genome-wide background fraction
        p_genome = (sub["meth_cat"] == "Hypomethylated").mean()

        for rank in RANKS:
            r = sub[sub["rank"] == rank]
            n_hypo  = (r["meth_cat"] == "Hypomethylated").sum()
            n_total = len(r)

            stat, p = proportions_ztest(count=n_hypo, nobs=n_total, value=p_genome)
            rows.append({
                "sample":   sample,
                "database": db,
                "rank":     rank,
                "n_total":  n_total,
                "n_hypo":   n_hypo,
                "pct_hypo": 100 * n_hypo / n_total,
                "p_genome": round(p_genome, 4),
                "z":        round(stat, 3),
                "p":        p,
            })

prop_df = pd.DataFrame(rows)
prop_df["sig"] = prop_df["p"].apply(
    lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
)
prop_df["enriched"] = prop_df.apply(
    lambda r: "enriched" if r["pct_hypo"] / 100 > r["p_genome"] else "depleted", axis=1
)
print(prop_df.to_string(index=False))



In [ ]:
from statsmodels.stats.multitest import multipletests

# FDR correction across all tests
prop_df["p_fdr"] = multipletests(prop_df["p"], method="fdr_bh")[1]
prop_df["sig_fdr"] = prop_df["p_fdr"].apply(
    lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
)
prop_df["enriched"] = prop_df.apply(
    lambda r: "enriched" if r["pct_hypo"] / 100 > r["p_genome"] else "depleted", axis=1
)

ENRICH_COLORS = {"enriched": "#c74467", "depleted": "#5663cc"}

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)

for row_i, sample in enumerate(SAMPLES):
    for col_j, db in enumerate(["G4Hunter", "Quadparser", "eG4"]):
        ax  = axes[row_i, col_j]
        sub = prop_df[(prop_df["sample"] == sample) & (prop_df["database"] == db)].sort_values("rank")

        p_bg = sub["p_genome"].iloc[0] * 100
        colors = [ENRICH_COLORS[e] for e in sub["enriched"]]

        bars = ax.bar(sub["rank"], sub["pct_hypo"], color=colors, width=0.7, alpha=0.85, zorder=2)
        ax.axhline(p_bg, color="black", lw=1.4, linestyle="--", zorder=3, label=f"Genome-wide ({p_bg:.1f}%)")
        ax.legend(fontsize=10, frameon=False)

        y_top = sub["pct_hypo"].max()
        for _, row in sub.iterrows():
            if row["sig_fdr"]:
                ax.text(
                    row["rank"], row["pct_hypo"] + y_top * 0.02,
                    row["sig_fdr"], ha="center", va="bottom", fontsize=9, fontweight="bold",
                )

        ax.set_xticks(sub["rank"])
        ax.tick_params(labelsize=13)
        ax.grid(axis="y", lw=0.4, alpha=0.6)
        ax.set_axisbelow(True)

        if row_i == 1:
            ax.set_xlabel("Replication Timing", fontsize=15)
        if col_j == 0:
            ax.set_ylabel("% Hypomethylated G4s", fontsize=15)
        if row_i == 0:
            ax.set_title(
                db, fontsize=15, fontweight="bold", y=1.06,
                bbox=dict(boxstyle="round,pad=0.45", facecolor="#d9d9d9", edgecolor="none", alpha=0.85),
            )

plt.tight_layout(rect=[0, 0, 0.94, 1])

pos0 = axes[0, 2].get_position()
pos1 = axes[1, 2].get_position()
mid  = (pos0.y0 + pos1.y1) / 2
box_x, box_w = pos0.x1 + 0.005, 0.028

for sample, box_y, box_h in zip(
    SAMPLES,
    [mid,           pos1.y0      ],
    [pos0.y1 - mid, mid - pos1.y0],
):
    fig.add_artist(FancyBboxPatch(
        (box_x, box_y), box_w, box_h,
        boxstyle="square,pad=0",
        facecolor="#d9d9d9", edgecolor="black", lw=1,
        transform=fig.transFigure, clip_on=False,
    ))
    fig.text(
        box_x + box_w / 2, box_y + box_h / 2,
        sample, fontsize=14, fontweight="bold",
        ha="center", va="center", rotation=270,
        transform=fig.transFigure,
    )

outdir = os.path.join(os.environ["SCRATCH"], "figures_g4_t2t")
os.makedirs(outdir, exist_ok=True)
fig.savefig(f"{outdir}/hypo_proportions_test.pdf", bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
from scipy.stats import kendalltau

hypo_pct = (
    pct_df[pct_df["meth_cat"] == "Hypomethylated"]
    .sort_values("rank")
)

rows = []
for sample in SAMPLES:
    for db in ["G4Hunter", "Quadparser", "eG4"]:
        sub = hypo_pct[(hypo_pct["sample"] == sample) & (hypo_pct["database"] == db)]
        tau, p = kendalltau(sub["rank"], sub["pct"])
        rows.append({"Sample": sample, "Database": db, "tau": round(tau, 3), "p": p})

stat_df = pd.DataFrame(rows)
stat_df["direction"] = stat_df["tau"].apply(lambda t: "increases" if t > 0 else "decreases")
stat_df["sig"] = stat_df["p"].apply(lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns")))
print(stat_df.to_string(index=False))



In [ ]:
from matplotlib.patches import FancyBboxPatch

locus_df["meth_cat"] = locus_df["avg_meth"].apply(map_meth)

COLOR_MAP = {"Hypomethylated": "#5663cc", 
             "Methylated": "#77777a", "Hypermethylated": "#a62145"}
counts = (
    locus_df
    .groupby(["sample", "rank", "database", "meth_cat"])
    .size()
    .reset_index(name="n")
)
totals = locus_df.groupby(["sample", "rank", "database"]).size().reset_index(name="total")
pct_df = counts.merge(totals, on=["sample", "rank", "database"])
pct_df["pct"] = 100 * pct_df["n"] / pct_df["total"]

RANKS = sorted(locus_df["rank"].unique())
x     = np.arange(len(RANKS))

fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharey=True, sharex=True)

for row_i, sample in enumerate(SAMPLES):
    for col_j, db in enumerate(DATABASES):
        ax  = axes[row_i, col_j]
        sub = pct_df[(pct_df["sample"] == sample) & (pct_df["database"] == db)]

        pivot = (
            sub.pivot_table(index="rank", columns="meth_cat", values="pct", fill_value=0)
               .reindex(columns=CATEGORIES, fill_value=0)
               .reindex(RANKS, fill_value=0)
        )

        bottom = np.zeros(len(RANKS))
        for cat in CATEGORIES:
            vals = pivot[cat].values
            ax.bar(x, vals, bottom=bottom, color=COLOR_MAP[cat], label=cat, width=0.65)
            bottom += vals

        for xi in x:
            ax.add_patch(plt.Rectangle(
                (xi - 0.325, 0), 0.65, 100,
                lw=1.3, edgecolor="black", facecolor="none",
                linestyle="-", zorder=3,
            ))

        ax.set_xticks(x)
        ax.set_xticklabels(RANKS, fontsize=15)
        ax.tick_params(axis="y", labelsize=15)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.grid(axis="y", lw=0.4, alpha=0.6)
        ax.set_ylim(0, 100)
        ax.set_axisbelow(True)

        if row_i == 1:
            ax.set_xlabel("Replication Timing", fontsize=18)
        if col_j == 0:
            ax.set_ylabel("% G4 Loci", fontsize=18)
        if row_i == 0:
            ax.set_title(
                db, fontsize=18, fontweight="bold",
                y=1.06,
                bbox=dict(boxstyle="round,pad=0.45", 
                          facecolor="#d9d9d9", 
                          edgecolor="black", 
                          alpha=0.85),
            )

handles = [plt.Rectangle((0, 0), 1, 1, color=COLOR_MAP[c]) for c in CATEGORIES]
fig.legend(
    handles, CATEGORIES,
    frameon=True,
    fancybox=True,
    shadow=True,
    loc="lower center", ncol=3, fontsize=16,
    bbox_to_anchor=(0.5, -0.03),
)

plt.tight_layout(rect=[0, 0.05, 0.94, 1])
pos0 = axes[0, 2].get_position()
pos1 = axes[1, 2].get_position()
mid  = (pos0.y0 + pos1.y1) / 2
box_x, box_w = pos0.x1, 0.032

for sample, box_y, box_h in zip(
    SAMPLES,
    [mid,           pos1.y0      ],
    [pos0.y1 - mid, mid - pos1.y0],
):
    fig.add_artist(FancyBboxPatch(
        (box_x, box_y), box_w, box_h,
        boxstyle="square,pad=0",
        facecolor="#d9d9d9", edgecolor="black", lw=1,
        transform=fig.transFigure, clip_on=False,
    ))
    fig.text(
        box_x + box_w / 2, box_y + box_h / 2,
        sample,
        fontsize=16, fontweight="bold",
        ha="center", va="center", rotation=270,
        transform=fig.transFigure,
    )

outdir = os.path.join(os.environ["SCRATCH"], "figures_g4_t2t")
os.makedirs(outdir, exist_ok=True)
fig.savefig(f"{outdir}/meth_categories_stacked.pdf", bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
f"{outdir}/meth_categories_stacked.pdf"

In [ ]:
from scipy.stats import mannwhitneyu

# _g4_id → is_eg4 mapping (merge on coordinates to survive bedtools sort)
eg4_flag = (g4_with_eg4_df[["seqID", "start", "end", "is_eg4"]]
    .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"}))
id_to_is_eg4 = (g4_df_id
    .merge(eg4_flag, on=["Chromosome", "Start", "End"], how="left")
    .set_index("_g4_id")["is_eg4"]
    .to_dict())

# annotate G4Hunter rows in locus_df
locus_g4h = locus_df[locus_df["database"] == "G4Hunter"].copy()
locus_g4h["is_eg4"] = locus_g4h["_g4_id"].map(id_to_is_eg4)

def _cliff(a, b):
    n1, n2 = len(a), len(b)
    if n1 == 0 or n2 == 0:
        return float("nan"), float("nan")
    stat, pval = mannwhitneyu(a, b, alternative="two-sided")
    return (2 * stat - n1 * n2) / (n1 * n2), pval

eg4_eff_records = []
for sample_name in SAMPLES:
    for rank in RANKS:
        sub = locus_g4h[(locus_g4h["sample"] == sample_name) & (locus_g4h["rank"] == rank)]
        a = sub.loc[sub["is_eg4"] == 1, "avg_meth"].dropna().values
        b = sub.loc[sub["is_eg4"] == 0, "avg_meth"].dropna().values
        delta, pval = _cliff(a, b)
        eg4_eff_records.append({
            "sample": sample_name, "rank": rank,
            "cliff_delta": delta, "pval": pval,
            "n_eg4": len(a), "n_non_eg4": len(b),
        })

eg4_eff_df = pd.DataFrame(eg4_eff_records)
eg4_eff_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

palette = {"eG4": "#d8e35d", "G4Hunter": "#ba8de0", "Quadparser": "#d35de3"}

fig, axes = plt.subplots(len(SAMPLES), 1, figsize=(10, 7.5), sharex=True)

for idx, (ax, sample_name) in enumerate(zip(axes, SAMPLES)):
    for database in ["G4Hunter", "Quadparser", "eG4"]:
        medians = [
            np.median(rep_meth_results[(database, sample_name, r)]["g4_meth"])
            if len(rep_meth_results[(database, sample_name, r)]["g4_meth"]) > 0
            else float("nan")
            for r in RANKS
        ]
        ax.plot(RANKS, medians, marker="o", linewidth=2,
                label=database, color=palette[database])

    if idx == 1:
        ax.set_xlabel("Replication Timing (Early → Late)", fontsize=16)
    else:
        ax.set_xlabel("")
    ax.set_ylabel("Median CpG Methylation", fontsize=16)
    ax.set_title(sample_name, fontsize=18,
                 bbox=dict(boxstyle="round,pad=0.4", facecolor="lightgray",
                           edgecolor="black", linewidth=1.5))
    ax.set_xticks(RANKS)
    ax.tick_params(labelsize=16)
    ax.grid(lw=0.4, alpha=0.6)
    sns.despine(ax=ax)

handles = [mpatches.Patch(color=palette[db], label=db) for db in palette]
axes[-1].legend(handles=handles, fontsize=13, frameon=True, shadow=True, fancybox=True)
plt.tight_layout()
fig.savefig(target_fig / "g4_meth_by_reptime.pdf", transparent=True, bbox_inches="tight")
fig.savefig(target_fig / "g4_meth_by_reptime.png", dpi=400, transparent=True, bbox_inches="tight")
plt.show()


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import pyranges as pr

WINDOW       = 2_000
HYPO_THRESH  = 0.2
HYPER_THRESH = 0.8
BIN_SIZE     = 50

state_palette = {
    "hypomethylated":  "#6a87de",
    "methylated":      "#e0a840",
    "hypermethylated": "crimson",
}

bin_edges  = np.arange(-WINDOW, WINDOW + BIN_SIZE, BIN_SIZE)
bin_labels = (bin_edges[:-1] + BIN_SIZE // 2).astype(int)

def _profile(df, bin_col):
    return (
        df.groupby(bin_col, observed=True)["methylation_level"]
        .agg(
            mean = "mean",
            q25  = lambda x: x.quantile(0.25),
            q75  = lambda x: x.quantile(0.75),
        )
        .reset_index()
        .rename(columns={bin_col: "dist"})
    )

flank_profiles   = {}
center_profiles  = {}
spearman_results = {}

for sample_name in SAMPLES:
    meth_df = (
        raw_meth[sample_name]
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
        [["Chromosome", "Start", "End", "methylation_level"]]
        .dropna(subset=["methylation_level"])
    )
    cpg_pr = pr.PyRanges(meth_df)
    eg4_pr = g4_prs["eG4"]

    locus_avg = (
        eg4_pr.join(cpg_pr).df
        .groupby("_g4_id")["methylation_level"]
        .mean().rename("avg_meth").reset_index()
    )
    locus_avg["state"] = "methylated"
    locus_avg.loc[locus_avg["avg_meth"] <  HYPO_THRESH,  "state"] = "hypomethylated"
    locus_avg.loc[locus_avg["avg_meth"] >= HYPER_THRESH, "state"] = "hypermethylated"

    eg4_classified = (
        eg4_pr.df
        .merge(locus_avg[["_g4_id", "state"]], on="_g4_id", how="inner")
        .assign(center   = lambda d: (d["Start"] + d["End"]) // 2,
                g4_start = lambda d: d["Start"],
                g4_end   = lambda d: d["End"])
    )

    for state in ["hypomethylated", "methylated", "hypermethylated"]:
        state_g4s = eg4_classified[eg4_classified["state"] == state].copy()
        if len(state_g4s) == 0:
            continue

        flank_df          = state_g4s[["Chromosome", "_g4_id", "center", "g4_start", "g4_end"]].copy()
        flank_df["Start"] = (state_g4s["g4_start"] - WINDOW).clip(lower=0).values
        flank_df["End"]   = (state_g4s["g4_end"]   + WINDOW).values
        flank_pr_tmp      = pr.PyRanges(flank_df)

        fj = flank_pr_tmp.join(cpg_pr)
        if len(fj.df) == 0:
            continue

        fj_df            = fj.df.copy()
        fj_df["cpg_mid"] = (fj_df["Start_b"] + fj_df["End_b"]) // 2

        fj_df["dist_bnd"] = np.where(
            fj_df["cpg_mid"] < fj_df["g4_start"],
            fj_df["cpg_mid"] - fj_df["g4_start"],
            np.where(
                fj_df["cpg_mid"] > fj_df["g4_end"],
                fj_df["cpg_mid"] - fj_df["g4_end"],
                0,
            )
        )
        fj_df["dist_ctr"] = fj_df["cpg_mid"] - fj_df["center"]

        fj_df["bin_bnd"] = pd.cut(fj_df["dist_bnd"], bins=bin_edges, labels=bin_labels)
        fj_df["bin_ctr"] = pd.cut(fj_df["dist_ctr"], bins=bin_edges, labels=bin_labels)

        flank_profiles[(sample_name, state)]  = _profile(fj_df, "bin_bnd")
        center_profiles[(sample_name, state)] = _profile(fj_df, "bin_ctr")

        flanking = fj_df[fj_df["dist_bnd"] != 0]
        if len(flanking) > 2:
            rho, pval = spearmanr(flanking["dist_bnd"].abs(), flanking["methylation_level"])
        else:
            rho, pval = float("nan"), float("nan")

        spearman_results[(sample_name, state)] = {
            "rho":              rho,
            "pval":             pval,
            "n_g4s":            len(state_g4s),
            "n_flanking_cpgs":  len(flanking),
        }



In [ ]:
states = ["hypomethylated", "methylated", "hypermethylated"]

fig, axes = plt.subplots(len(states), len(SAMPLES),
                         figsize=(6 * len(SAMPLES), 3.5 * len(states)),
                         sharey="row")

for row, state in enumerate(states):
    color = state_palette[state]
    for col, sample_name in enumerate(SAMPLES):
        ax = axes[row, col]
        if (sample_name, state) not in flank_profiles:
            continue
        prof = flank_profiles[(sample_name, state)].copy()
        prof["dist"] = prof["dist"].astype(float)
        prof = prof.dropna(subset=["dist", "mean", "q25", "q75"])

        center_val       = prof.loc[prof["dist"].abs().idxmin(), "mean"]
        prof["rel_mean"] = (prof["mean"] - center_val) / center_val * 100
        iqr_half         = (prof["q75"] - prof["q25"]) / 2 / abs(center_val) * 100

        ax.plot(prof["dist"], prof["rel_mean"], color=color, linewidth=2)
        ax.fill_between(prof["dist"],
                        prof["rel_mean"] - iqr_half,
                        prof["rel_mean"] + iqr_half,
                        color=color, alpha=0.25)

        ax.axvline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)
        ax.axhline(0, color="black", linewidth=0.6, linestyle=":", alpha=0.4)
        ax.tick_params(labelsize=12)
        ax.grid(lw=0.4, alpha=0.6)
        sns.despine(ax=ax)

        if row == 0:
            ax.set_title(sample_name, fontsize=16,
                         bbox=dict(boxstyle="round,pad=0.4", facecolor="lightgray",
                                   edgecolor="black", linewidth=1.5))
        if col == 0:
            ax.set_ylabel(f"{state}\nRelative Δ (%)", fontsize=12, color=color)
        if row == len(states) - 1:
            ax.set_xlabel("Distance from eG4 boundary (bp)", fontsize=13)

        sr = spearman_results[(sample_name, state)]
        ax.text(0.97, 0.05, f"ρ={sr['rho']:.3f}  p={sr['pval']:.2e}",
                transform=ax.transAxes, ha="right", fontsize=10, style="italic")

plt.tight_layout()
fig.savefig(target_fig / "eg4_flanking_meth_by_state.pdf", transparent=True, bbox_inches="tight")
fig.savefig(target_fig / "eg4_flanking_meth_by_state.png", dpi=400, transparent=True, bbox_inches="tight")
plt.show()


## Classify by its immediate surroundings

In [ ]:
control_df = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_HG002_G4Hunter_filtered_with_meth.cpg_strict.no_rank.filtered.tsv.gz",
usecols=["seqID", "control_start", "control_end", "control_methylation_level"])
control_df


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import pyranges as pr

WINDOW       = 2_000
FLANK_CLASS  = 20      # bp around boundaries used for classification
HYPO_THRESH  = 0.2
HYPER_THRESH = 0.8
BIN_SIZE     = 50

state_palette = {
    "Hypomethylated":  "#6a87de",
    "Methylated":      "#e0a840",
    "Hypermethylated": "crimson",
}

bin_edges  = np.arange(-WINDOW, WINDOW + BIN_SIZE, BIN_SIZE)
bin_labels = (bin_edges[:-1] + BIN_SIZE // 2).astype(int)

def _profile(df, bin_col):
    return (
        df.groupby(bin_col, observed=True)["methylation_level"]
        .agg(
            mean = "mean",
            q25  = lambda x: x.quantile(0.25),
            q75  = lambda x: x.quantile(0.75),
        )
        .reset_index()
        .rename(columns={bin_col: "dist"})
    )

flank_profiles   = {}
center_profiles  = {}
spearman_results = {}

for sample_name in SAMPLES:
    meth_df = (
        raw_meth[sample_name]
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
        [["Chromosome", "Start", "End", "methylation_level"]]
        .dropna(subset=["methylation_level"])
    )
    cpg_pr = pr.PyRanges(meth_df)
    eg4_pr = g4_prs["eG4"]

    # classify by average methylation in ±FLANK_CLASS bp window
    class_df          = eg4_pr.df[["Chromosome", "Start", "End", "_g4_id"]].copy()
    class_df["Start"] = (class_df["Start"] - FLANK_CLASS).clip(lower=0)
    class_df["End"]   = class_df["End"] + FLANK_CLASS
    class_pr          = pr.PyRanges(class_df)

    joined_class = class_pr.join(cpg_pr)
    locus_avg = (
        joined_class.df
        .groupby("_g4_id")["methylation_level"]
        .mean().rename("avg_meth").reset_index()
    )
    locus_avg["state"] = "Methylated"
    locus_avg.loc[locus_avg["avg_meth"] <  HYPO_THRESH,  "state"] = "Hypomethylated"
    locus_avg.loc[locus_avg["avg_meth"] >= HYPER_THRESH, "state"] = "Hypermethylated"

    # all eG4s — those without any CpG in ±20bp get dropped (no methylation signal)
    eg4_classified = (
        eg4_pr.df
        .merge(locus_avg[["_g4_id", "state"]], on="_g4_id", how="inner")
        .assign(center   = lambda d: (d["Start"] + d["End"]) // 2,
                g4_start = lambda d: d["Start"],
                g4_end   = lambda d: d["End"])
    )

    for state in ["Hypomethylated", "Methylated", "Hypermethylated"]:
        state_g4s = eg4_classified[eg4_classified["state"] == state].copy()
        if len(state_g4s) == 0:
            continue

        flank_df          = state_g4s[["Chromosome", "_g4_id", "center", "g4_start", "g4_end"]].copy()
        flank_df["Start"] = (state_g4s["g4_start"] - WINDOW).clip(lower=0).values
        flank_df["End"]   = (state_g4s["g4_end"]   + WINDOW).values
        flank_pr_tmp      = pr.PyRanges(flank_df)

        fj = flank_pr_tmp.join(cpg_pr)
        if len(fj.df) == 0:
            continue

        fj_df            = fj.df.copy()
        fj_df["cpg_mid"] = (fj_df["Start_b"] + fj_df["End_b"]) // 2

        fj_df["dist_bnd"] = np.where(
            fj_df["cpg_mid"] < fj_df["g4_start"],
            fj_df["cpg_mid"] - fj_df["g4_start"],
            np.where(
                fj_df["cpg_mid"] > fj_df["g4_end"],
                fj_df["cpg_mid"] - fj_df["g4_end"],
                0,
            )
        )
        fj_df["dist_ctr"] = fj_df["cpg_mid"] - fj_df["center"]

        fj_df["bin_bnd"] = pd.cut(fj_df["dist_bnd"], bins=bin_edges, labels=bin_labels)
        fj_df["bin_ctr"] = pd.cut(fj_df["dist_ctr"], bins=bin_edges, labels=bin_labels)

        flank_profiles[(sample_name, state)]  = _profile(fj_df, "bin_bnd")
        center_profiles[(sample_name, state)] = _profile(fj_df, "bin_ctr")

        flanking = fj_df[fj_df["dist_bnd"] != 0]
        if len(flanking) > 2:
            rho, pval = spearmanr(flanking["dist_bnd"].abs(), flanking["methylation_level"])
        else:
            rho, pval = float("nan"), float("nan")

        spearman_results[(sample_name, state)] = {
            "rho":             rho,
            "pval":            pval,
            "n_g4s":           len(state_g4s),
            "n_flanking_cpgs": len(flanking),
        }


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

states = ["Hypomethylated", "Methylated", "Hypermethylated"]
fig, axes = plt.subplots(len(states), len(SAMPLES),
                         figsize=(6 * len(SAMPLES), 3.5 * len(states)),
                         sharey="row")

for row, state in enumerate(states):
    color = state_palette[state]
    for col, sample_name in enumerate(SAMPLES):
        ax = axes[row, col]
        if (sample_name, state) not in flank_profiles:
            ax.set_visible(False)
            continue

        prof = flank_profiles[(sample_name, state)].copy()
        prof["dist"] = prof["dist"].astype(float)
        prof = prof.dropna(subset=["dist", "mean", "q25", "q75"])

        center_val       = prof.loc[prof["dist"].abs().idxmin(), "mean"]
        prof["rel_mean"] = (prof["mean"] - center_val) / center_val * 100
        iqr_half         = (prof["q75"] - prof["q25"]) / 2 / abs(center_val) * 100

        ax.plot(prof["dist"], prof["rel_mean"], color=color, linewidth=2)
        ax.fill_between(prof["dist"],
                        prof["rel_mean"] - iqr_half,
                        prof["rel_mean"] + iqr_half,
                        color=color, alpha=0.25)
        ax.axvline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)
        ax.axhline(0, color="black", linewidth=0.6, linestyle=":", alpha=0.4)

        sr = spearman_results[(sample_name, state)]
        ax.text(0.97, 0.05, f"ρ={sr['rho']:.3f}  p={sr['pval']:.2e}\nn={sr['n_g4s']:,}",
                transform=ax.transAxes, ha="right", fontsize=9, style="italic")

        if row == 0:
            ax.set_title(sample_name, fontsize=16,
                         bbox=dict(boxstyle="round,pad=0.4", facecolor="lightgray",
                                   edgecolor="black", linewidth=1.5))
        if col == 0:
            ax.set_ylabel(f"{state}\nRelative Δ (%)", fontsize=12, color=color)
        if row == len(states) - 1:
            ax.set_xlabel("Distance from eG4 boundary (bp)", fontsize=13)

        ax.tick_params(labelsize=11)
        ax.grid(lw=0.4, alpha=0.6)
        sns.despine(ax=ax)

plt.tight_layout()
fig.savefig(target_fig / "eg4_flanking_meth_by_state.pdf", transparent=True, bbox_inches="tight")
fig.savefig(target_fig / "eg4_flanking_meth_by_state.png", dpi=400, transparent=True, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.stats import spearmanr

SAMPLE = "HG002"   # controls are matched to this sample
HYPER_THRESH = 0.8
WINDOW       = 2_000
FLANK_CLASS  = 20      # bp around boundaries used for classification
HYPO_THRESH  = 0.2
HYPER_THRESH = 0.8
BIN_SIZE     = 50

state_palette = {
    "Hypomethylated":  "#6a87de",
    "Methylated":      "#e0a840",
    "Hypermethylated": "crimson",
}

bin_edges  = np.arange(-WINDOW, WINDOW + BIN_SIZE, BIN_SIZE)
bin_labels = (bin_edges[:-1] + BIN_SIZE // 2).astype(int)
# reuse meth data
meth_df = (
    raw_meth[SAMPLE]
    .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
    [["Chromosome", "Start", "End", "methylation_level"]]
    .dropna(subset=["methylation_level"])
)
cpg_pr = pr.PyRanges(meth_df)

# prep controls
ctrl = (
    control_df
    .rename(columns={"seqID": "Chromosome", "control_start": "Start", "control_end": "End"})
    .drop_duplicates(subset=["Chromosome", "Start", "End"])
    .reset_index(drop=True)
)
ctrl["_ctrl_id"] = ctrl.index
ctrl["g4_start"]  = ctrl["Start"]
ctrl["g4_end"]    = ctrl["End"]

# ── optional: keep only isolated eG4s (>1000bp from any other eG4) ────────────
from pybedtools import BedTool
ISOLATION_DIST = 1_000
WINDOW = 2_000
eg4_bed = BedTool.from_dataframe(
    eg4_df[["seqID", "start", "end"]].drop_duplicates()
    .rename(columns={"seqID": "Chromosome"})
).sort()
closest_df = eg4_bed.closest(eg4_bed, io=True, D="ref").to_dataframe(
    names=["chrom", "start", "end", "chrom2", "start2", "end2", "dist"]
)
closest_df["dist"] = closest_df["dist"].astype(int)
isolated_eg4_set = set(
    zip(closest_df.query("dist.abs() > @ISOLATION_DIST | dist == -1")["chrom"],
        closest_df.query("dist.abs() > @ISOLATION_DIST | dist == -1")["start"],
        closest_df.query("dist.abs() > @ISOLATION_DIST | dist == -1")["end"])
)
print(f"Isolated eG4s (>{ISOLATION_DIST}bp from neighbor): {len(isolated_eg4_set):,}")
# ─────────────────────────────────────────────────────────────────────────────

# classify controls by ±FLANK_CLASS bp flank meth
class_ctrl          = ctrl[["Chromosome", "Start", "End", "_ctrl_id"]].copy()
class_ctrl["Start"] = (ctrl["Start"] - FLANK_CLASS).clip(lower=0)
class_ctrl["End"]   = ctrl["End"] + FLANK_CLASS
locus_avg_ctrl = (
    pr.PyRanges(class_ctrl).join(cpg_pr).df
    .groupby("_ctrl_id")["methylation_level"]
    .mean().rename("avg_meth").reset_index()
)
locus_avg_ctrl["state"] = "Methylated"
locus_avg_ctrl.loc[locus_avg_ctrl["avg_meth"] <  HYPO_THRESH,  "state"] = "Hypomethylated"
locus_avg_ctrl.loc[locus_avg_ctrl["avg_meth"] >= HYPER_THRESH, "state"] = "Hypermethylated"

ctrl_classified = ctrl.merge(locus_avg_ctrl[["_ctrl_id", "state"]], on="_ctrl_id", how="inner")
print(ctrl_classified["state"].value_counts())

flank_profiles_ctrl   = {}
spearman_results_ctrl = {}

for state in ["Hypomethylated", "Methylated", "Hypermethylated"]:
    sub = ctrl_classified[ctrl_classified["state"] == state]
    if len(sub) == 0:
        continue
    flank_df          = sub[["Chromosome", "_ctrl_id", "g4_start", "g4_end"]].copy()
    flank_df["Start"] = (sub["g4_start"] - WINDOW).clip(lower=0).values
    flank_df["End"]   = (sub["g4_end"]   + WINDOW).values
    fj = pr.PyRanges(flank_df).join(cpg_pr)
    if len(fj.df) == 0:
        continue

    fj_df            = fj.df.copy()
    fj_df["cpg_mid"] = (fj_df["Start_b"] + fj_df["End_b"]) // 2
    fj_df["dist_bnd"] = np.where(
        fj_df["cpg_mid"] < fj_df["g4_start"],
        fj_df["cpg_mid"] - fj_df["g4_start"],
        np.where(fj_df["cpg_mid"] > fj_df["g4_end"],
                 fj_df["cpg_mid"] - fj_df["g4_end"], 0),
    )
    fj_df["bin_bnd"] = pd.cut(fj_df["dist_bnd"], bins=bin_edges, labels=bin_labels)
    flank_profiles_ctrl[state] = _profile(fj_df, "bin_bnd")

    flanking = fj_df[fj_df["dist_bnd"] != 0]
    rho, pval = (spearmanr(flanking["dist_bnd"].abs(), flanking["methylation_level"])
                 if len(flanking) > 2 else (float("nan"), float("nan")))
    spearman_results_ctrl[state] = {"rho": rho, "pval": pval, "n_ctrl": len(sub)}
    print(f"{state}: {len(sub):,} controls | ρ={rho:.3f}, p={pval:.2e}")


In [ ]:
def _plot_rel(ax, prof_dict, key, color, label, ls="-"):
    if key not in prof_dict:
        return
    prof = prof_dict[key].copy()
    prof["dist"] = prof["dist"].astype(float)
    prof = prof.dropna(subset=["dist", "mean"])
    c0 = prof.loc[prof["dist"].abs().idxmin(), "mean"]
    if c0 == 0 or np.isnan(c0):
        return
    rel     = (prof["mean"] - c0) / c0 * 100
    iqr_h   = (prof["q75"] - prof["q25"]) / 2 / abs(c0) * 100
    ax.plot(prof["dist"], rel, color=color, lw=2.5, ls=ls, label=label)
    ax.fill_between(prof["dist"], rel - iqr_h, rel + iqr_h, alpha=0.15, color=color)

states = ["Hypomethylated", "Methylated", "Hypermethylated"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, state in enumerate(states):
    ax = axes[i]
    color = state_palette[state]
    _plot_rel(ax, flank_profiles,      (SAMPLE, state), color,  "eG4",    ls="-")
    _plot_rel(ax, flank_profiles_ctrl, state,           "gray", "Control", ls="--")
    ax.axvline(0, color="black", lw=1,   ls="--", alpha=0.5)
    ax.axhline(0, color="black", lw=0.6, ls=":",  alpha=0.4)
    ax.set_title(state, fontsize=16, color=color)
    ax.set_xlabel("Distance from boundary (bp)", fontsize=14)
    ax.tick_params(labelsize=12)
    ax.grid(lw=0.4, alpha=0.6)
    if i == 0:
        ax.set_ylabel("Relative Δ (%)", fontsize=14)
        ax.legend(fontsize=12, frameon=False)

fig.tight_layout()
fig.savefig(target_fig / "eg4_vs_ctrl_flanking_meth_by_state.pdf",
            bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

STATE  = "Hypomethylated"

# ── eG4: re-intersect for this state, get per-locus median per bin ────────────
eg4_state           = eg4_classified[eg4_classified["state"] == STATE].copy()
flank_eg4           = eg4_state[["Chromosome", "_g4_id", "g4_start", "g4_end"]].copy()
flank_eg4["Start"]  = (eg4_state["g4_start"] - WINDOW).clip(lower=0).values
flank_eg4["End"]    = (eg4_state["g4_end"]   + WINDOW).values

fj_eg4             = pr.PyRanges(flank_eg4).join(cpg_pr).df.copy()
fj_eg4["cpg_mid"]  = (fj_eg4["Start_b"] + fj_eg4["End_b"]) // 2
fj_eg4["dist_bnd"] = np.where(
    fj_eg4["cpg_mid"] < fj_eg4["g4_start"],
    fj_eg4["cpg_mid"] - fj_eg4["g4_start"],
    np.where(fj_eg4["cpg_mid"] > fj_eg4["g4_end"],
             fj_eg4["cpg_mid"] - fj_eg4["g4_end"], 0),
)
fj_eg4["bin_bnd"] = pd.cut(fj_eg4["dist_bnd"], bins=bin_edges, labels=bin_labels)
eg4_locus = (fj_eg4.groupby(["_g4_id", "bin_bnd"], observed=True)["methylation_level"]
             .median().reset_index())

# ── control: same ─────────────────────────────────────────────────────────────
ctrl_state          = ctrl_classified[ctrl_classified["state"] == STATE].copy()
flank_ctrl          = ctrl_state[["Chromosome", "_ctrl_id", "g4_start", "g4_end"]].copy()
flank_ctrl["Start"] = (ctrl_state["g4_start"] - WINDOW).clip(lower=0).values
flank_ctrl["End"]   = (ctrl_state["g4_end"]   + WINDOW).values

fj_ctrl             = pr.PyRanges(flank_ctrl).join(cpg_pr).df.copy()
fj_ctrl["cpg_mid"]  = (fj_ctrl["Start_b"] + fj_ctrl["End_b"]) // 2
fj_ctrl["dist_bnd"] = np.where(
    fj_ctrl["cpg_mid"] < fj_ctrl["g4_start"],
    fj_ctrl["cpg_mid"] - fj_ctrl["g4_start"],
    np.where(fj_ctrl["cpg_mid"] > fj_ctrl["g4_end"],
             fj_ctrl["cpg_mid"] - fj_ctrl["g4_end"], 0),
)
fj_ctrl["bin_bnd"] = pd.cut(fj_ctrl["dist_bnd"], bins=bin_edges, labels=bin_labels)
ctrl_locus = (fj_ctrl.groupby(["_ctrl_id", "bin_bnd"], observed=True)["methylation_level"]
              .median().reset_index())

# ── per-bin MWU → BH FDR ─────────────────────────────────────────────────────
bins = sorted(
    set(eg4_locus["bin_bnd"].astype(float).unique()) |
    set(ctrl_locus["bin_bnd"].astype(float).unique())
)
pvals = []
for b in bins:
    e = eg4_locus[eg4_locus["bin_bnd"].astype(float) == b]["methylation_level"].dropna()
    c = ctrl_locus[ctrl_locus["bin_bnd"].astype(float) == b]["methylation_level"].dropna()
    if len(e) < 5 or len(c) < 5:
        pvals.append(np.nan); continue
    _, p = mannwhitneyu(e, c, alternative="two-sided")
    pvals.append(p)

pvals_arr = np.array(pvals, dtype=float)
mask = ~np.isnan(pvals_arr)
if mask.sum() > 0:
    _, pvals_corr, _, _ = multipletests(pvals_arr[mask], method="fdr_bh")
    pvals_arr[mask] = pvals_corr

sig_df = pd.DataFrame({"bin": bins, "fdr": pvals_arr})
sig_df["sig"] = sig_df["fdr"] < 0.05
print(f"Significant bins: {sig_df['sig'].sum()} / {len(sig_df)}")

# ── plot: profile + significance track ───────────────────────────────────────
color = state_palette[STATE]
fig, (ax_prof, ax_sig) = plt.subplots(
    2, 1, figsize=(14, 6), sharex=True,
    gridspec_kw={"height_ratios": [4, 1]}
)

def _plot_raw(ax, fj, color, label, ls="-"):
    prof = (fj.groupby("bin_bnd", observed=True)["methylation_level"]
            .agg(mean="mean",
                 q25=lambda x: x.quantile(0.25),
                 q75=lambda x: x.quantile(0.75))
            .reset_index())
    prof["dist"] = prof["bin_bnd"].astype(float)
    prof = prof.dropna(subset=["dist", "mean"])
    ax.plot(prof["dist"], prof["mean"], color=color, lw=2.5, ls=ls, label=label)
    ax.fill_between(prof["dist"], prof["q25"], prof["q75"], alpha=0.15, color=color)

_plot_raw(ax_prof, fj_eg4,  color,  "eG4",     ls="-")
_plot_raw(ax_prof, fj_ctrl, "gray", "Control", ls="--")
ax_prof.axvline(0, color="black", lw=1, ls="--", alpha=0.5)
ax_prof.set_ylabel("Median CpG Methylation", fontsize=14)
ax_prof.set_ylim(0, 1)
ax_prof.legend(fontsize=12, frameon=False)
ax_prof.grid(lw=0.4, alpha=0.6)
ax_prof.set_title(STATE, fontsize=16, color=color)
ax_prof.tick_params(labelsize=12)

ax_sig.bar(sig_df["bin"], -np.log10(sig_df["fdr"].clip(lower=1e-300)),
           width=BIN_SIZE * 0.9,
           color=[color if s else "lightgray" for s in sig_df["sig"]])
ax_sig.axhline(-np.log10(0.05), color="black", lw=1, ls=":", alpha=0.7, label="FDR=0.05")
ax_sig.axvline(0, color="black", lw=1, ls="--", alpha=0.5)
ax_sig.set_ylabel("-log₁₀(FDR)", fontsize=12)
ax_sig.set_xlabel("Distance from eG4 boundary (bp)", fontsize=14)
ax_sig.tick_params(labelsize=11)
ax_sig.grid(lw=0.4, alpha=0.6)

fig.tight_layout()
fig.savefig(target_fig / f"eg4_vs_ctrl_{STATE.lower()}_mwu.pdf",
            bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
comp_map = (
    eg4_in_comp_df[["Chromosome", "g4_Start", "g4_End", "group"]]
    .rename(columns={"g4_Start": "Start", "g4_End": "End"})
    .drop_duplicates()
)

eg4_with_comp = (
    eg4_classified[eg4_classified["state"] == STATE]
    .merge(comp_map, on=["Chromosome", "Start", "End"], how="left")
)

print(eg4_with_comp["group"].value_counts())
print(f"Unmatched: {eg4_with_comp['group'].isna().sum()}")


In [ ]:
# per-compartment MWU: hypomethylated eG4 vs matched controls
# uses eg4_in_comp_df (already computed) to get compartment labels per eG4

eg4_with_comp = (
    eg4_classified[eg4_classified["state"] == STATE]
    .merge(
        eg4_in_comp_df[["Chromosome", "Start", "End", "group"]],
        left_on=["Chromosome", "Start", "End"], right_on=["Chromosome", "Start", "End"],
        how="left"
    )
)

results_by_comp = {}

for comp, sub_eg4 in eg4_with_comp.groupby("group"):
    if len(sub_eg4) < 50:
        continue

    flank           = sub_eg4[["Chromosome", "_g4_id", "g4_start", "g4_end"]].copy()
    flank["Start"]  = (sub_eg4["g4_start"] - WINDOW).clip(lower=0).values
    flank["End"]    = (sub_eg4["g4_end"]   + WINDOW).values

    fj = pr.PyRanges(flank).join(cpg_pr).df.copy()
    if len(fj) == 0:
        continue
    fj["cpg_mid"]  = (fj["Start_b"] + fj["End_b"]) // 2
    fj["dist_bnd"] = np.where(
        fj["cpg_mid"] < fj["g4_start"], fj["cpg_mid"] - fj["g4_start"],
        np.where(fj["cpg_mid"] > fj["g4_end"], fj["cpg_mid"] - fj["g4_end"], 0)
    )
    fj["bin_bnd"] = pd.cut(fj["dist_bnd"], bins=bin_edges, labels=bin_labels)
    eg4_loc = (fj.groupby(["_g4_id", "bin_bnd"], observed=True)["methylation_level"]
               .median().reset_index())

    # compare against the FULL control hypomethylated set (no compartment split for controls)
    pvals = []
    for b in bins:
        e = eg4_loc[eg4_loc["bin_bnd"].astype(float) == b]["methylation_level"].dropna()
        c = ctrl_locus[ctrl_locus["bin_bnd"].astype(float) == b]["methylation_level"].dropna()
        if len(e) < 5 or len(c) < 5:
            pvals.append(np.nan); continue
        _, p = mannwhitneyu(e, c, alternative="two-sided")
        pvals.append(p)

    pvals_arr = np.array(pvals, dtype=float)
    mask = ~np.isnan(pvals_arr)
    if mask.sum() > 0:
        _, corr, _, _ = multipletests(pvals_arr[mask], method="fdr_bh")
        pvals_arr[mask] = corr

    n_sig = np.sum(pvals_arr < 0.05)
    results_by_comp[comp] = {"pvals": pvals_arr, "n_sig": n_sig, "n_eg4": len(sub_eg4), "fj": fj}
    print(f"{comp:30s}  n={len(sub_eg4):5,}  sig_bins={n_sig}")


In [ ]:
target_fig / f"eg4_vs_ctrl_{STATE.lower()}_mwu.pdf"

In [ ]:
def _plot_abs(ax, prof_dict, key, color, label, ls="-"):
    if key not in prof_dict:
        return
    prof = prof_dict[key].copy()
    prof["dist"] = prof["dist"].astype(float)
    prof = prof.dropna(subset=["dist", "mean"])
    ax.plot(prof["dist"], prof["mean"], color=color, lw=2.5, ls=ls, label=label)
    ax.fill_between(prof["dist"], prof["q25"], prof["q75"], alpha=0.15, color=color)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, state in enumerate(["Hypomethylated", "Methylated", "Hypermethylated"]):
    ax = axes[i]
    color = state_palette[state]
    _plot_abs(ax, flank_profiles,      (SAMPLE, state), color,  "eG4",     ls="-")
    _plot_abs(ax, flank_profiles_ctrl, state,           "gray", "Control", ls="--")
    ax.axvline(0, color="black", lw=1, ls="--", alpha=0.5)
    ax.set_title(state, fontsize=16, color=color)
    ax.set_xlabel("Distance from boundary (bp)", fontsize=14)
    ax.set_ylim(0, 1)
    ax.tick_params(labelsize=12)
    ax.grid(lw=0.4, alpha=0.6)
    if i == 0:
        ax.set_ylabel("Median CpG Methylation", fontsize=14)
        ax.legend(fontsize=12, frameon=False)

fig.tight_layout()
fig.savefig(target_fig / "eg4_vs_ctrl_flanking_meth_absolute.pdf",
            bbox_inches="tight", transparent=True)
plt.show()


## Replication Timing

In [ ]:
flank_profiles_rep   = {}   # (sample_name, rank, state)
spearman_results_rep = {}

for sample_name in SAMPLES:
    meth_df = (
        raw_meth[sample_name]
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
        [["Chromosome", "Start", "End", "methylation_level"]]
        .dropna(subset=["methylation_level"])
    )
    cpg_pr = pr.PyRanges(meth_df)
    eg4_pr = g4_prs["eG4"]

    locus_avg = (
        eg4_pr.join(cpg_pr).df
        .groupby("_g4_id")["methylation_level"]
        .mean().rename("avg_meth").reset_index()
    )
    locus_avg["state"] = "methylated"
    locus_avg.loc[locus_avg["avg_meth"] <  HYPO_THRESH,  "state"] = "hypomethylated"
    locus_avg.loc[locus_avg["avg_meth"] >= HYPER_THRESH, "state"] = "hypermethylated"

    eg4_classified = (
        eg4_pr.df
        .merge(locus_avg[["_g4_id", "state"]], on="_g4_id", how="inner")
        .assign(center   = lambda d: (d["Start"] + d["End"]) // 2,
                g4_start = lambda d: d["Start"],
                g4_end   = lambda d: d["End"])
    )

    for rank in RANKS:
        eg4_in_decile = pr.PyRanges(eg4_classified).overlap(rep_prs[rank]).df

        for state in ["hypomethylated", "methylated", "hypermethylated"]:
            state_g4s = eg4_in_decile[eg4_in_decile["state"] == state].copy()
            if len(state_g4s) == 0:
                continue

            flank_df          = state_g4s[["Chromosome", "_g4_id", "center", "g4_start", "g4_end"]].copy()
            flank_df["Start"] = (state_g4s["g4_start"] - WINDOW).clip(lower=0).values
            flank_df["End"]   = (state_g4s["g4_end"]   + WINDOW).values
            flank_pr_tmp      = pr.PyRanges(flank_df)

            fj = flank_pr_tmp.join(cpg_pr)
            if len(fj.df) == 0:
                continue

            fj_df            = fj.df.copy()
            fj_df["cpg_mid"] = (fj_df["Start_b"] + fj_df["End_b"]) // 2
            fj_df["dist_bnd"] = np.where(
                fj_df["cpg_mid"] < fj_df["g4_start"],
                fj_df["cpg_mid"] - fj_df["g4_start"],
                np.where(
                    fj_df["cpg_mid"] > fj_df["g4_end"],
                    fj_df["cpg_mid"] - fj_df["g4_end"],
                    0,
                )
            )
            fj_df["bin_bnd"] = pd.cut(fj_df["dist_bnd"], bins=bin_edges, labels=bin_labels)
            flank_profiles_rep[(sample_name, rank, state)] = _profile(fj_df, "bin_bnd")

            flanking = fj_df[fj_df["dist_bnd"] != 0]
            if len(flanking) > 2:
                rho, pval = spearmanr(flanking["dist_bnd"].abs(), flanking["methylation_level"])
            else:
                rho, pval = float("nan"), float("nan")

            spearman_results_rep[(sample_name, rank, state)] = {
                "rho": rho, "pval": pval,
                "n_g4s": len(state_g4s), "n_flanking_cpgs": len(flanking),
            }


In [ ]:
states = ["hypomethylated", "methylated", "hypermethylated"]

for sample_name in SAMPLES:
    fig, axes = plt.subplots(
        len(states), len(RANKS),
        figsize=(2.8 * len(RANKS), 3.2 * len(states)),
        sharey="row",
    )
    for row, state in enumerate(states):
        color = state_palette[state]
        for col, rank in enumerate(RANKS):
            ax = axes[row, col]
            if (sample_name, rank, state) not in flank_profiles_rep:
                ax.set_visible(False)
                continue

            prof = flank_profiles_rep[(sample_name, rank, state)].copy()
            prof["dist"] = prof["dist"].astype(float)
            prof = prof.dropna(subset=["dist", "mean", "q25", "q75"])

            center_val       = prof.loc[prof["dist"].abs().idxmin(), "mean"]
            prof["rel_mean"] = (prof["mean"] - center_val) / center_val * 100
            iqr_half         = (prof["q75"] - prof["q25"]) / 2 / abs(center_val) * 100

            ax.plot(prof["dist"], prof["rel_mean"], color=color, linewidth=1.5)
            ax.fill_between(prof["dist"],
                            prof["rel_mean"] - iqr_half,
                            prof["rel_mean"] + iqr_half,
                            color=color, alpha=0.25)
            ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
            ax.axhline(0, color="black", linewidth=0.5, linestyle=":", alpha=0.4)
            ax.tick_params(labelsize=8)
            ax.grid(lw=0.4, alpha=0.6)
            sns.despine(ax=ax)

            sr = spearman_results_rep[(sample_name, rank, state)]
            ax.set_title(f"Rank {rank}\nρ={sr['rho']:.2f}", fontsize=8, color=color)

            if col == 0:
                ax.set_ylabel(f"{state}\nRelative Δ (%)", fontsize=9, color=color)
            if row == len(states) - 1:
                ax.set_xlabel("Distance (bp)", fontsize=8)

    plt.suptitle(sample_name, fontsize=14, y=1.01)
    plt.tight_layout()
    fig.savefig(target_fig / f"eg4_flanking_meth_reptime_{sample_name}.pdf",
                transparent=True, bbox_inches="tight")
    fig.savefig(target_fig / f"eg4_flanking_meth_reptime_{sample_name}.png",
                dpi=400, transparent=True, bbox_inches="tight")
    plt.show()


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
import pyranges as pr

WINDOW   = 2_000
BIN_SIZE = 50

bin_edges  = np.arange(-WINDOW, WINDOW + BIN_SIZE, BIN_SIZE)
bin_labels = (bin_edges[:-1] + BIN_SIZE // 2).astype(int)

def _profile(df, bin_col):
    return (
        df.groupby(bin_col, observed=True)["methylation_level"]
        .agg(
            mean = "mean",
            q25  = lambda x: x.quantile(0.25),
            q75  = lambda x: x.quantile(0.75),
        )
        .reset_index()
        .rename(columns={bin_col: "dist"})
    )

flank_profiles_all   = {}
center_profiles_all  = {}
spearman_results_all = {}

for sample_name in SAMPLES:
    meth_df = (
        raw_meth[sample_name]
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
        [["Chromosome", "Start", "End", "methylation_level"]]
        .dropna(subset=["methylation_level"])
    )
    cpg_pr = pr.PyRanges(meth_df)
    eg4_pr = g4_prs["eG4"]

    eg4_coords = eg4_pr.df.assign(
        center   = lambda d: (d["Start"] + d["End"]) // 2,
        g4_start = lambda d: d["Start"],
        g4_end   = lambda d: d["End"],
    )

    flank_df          = eg4_coords[["Chromosome", "_g4_id", "center", "g4_start", "g4_end"]].copy()
    flank_df["Start"] = (eg4_coords["g4_start"] - WINDOW).clip(lower=0).values
    flank_df["End"]   = (eg4_coords["g4_end"]   + WINDOW).values
    flank_pr_tmp      = pr.PyRanges(flank_df)

    fj = flank_pr_tmp.join(cpg_pr)
    if len(fj.df) == 0:
        continue

    fj_df            = fj.df.copy()
    fj_df["cpg_mid"] = (fj_df["Start_b"] + fj_df["End_b"]) // 2

    fj_df["dist_bnd"] = np.where(
        fj_df["cpg_mid"] < fj_df["g4_start"],
        fj_df["cpg_mid"] - fj_df["g4_start"],
        np.where(
            fj_df["cpg_mid"] > fj_df["g4_end"],
            fj_df["cpg_mid"] - fj_df["g4_end"],
            0,
        )
    )
    fj_df["dist_ctr"] = fj_df["cpg_mid"] - fj_df["center"]

    fj_df["bin_bnd"] = pd.cut(fj_df["dist_bnd"], bins=bin_edges, labels=bin_labels)
    fj_df["bin_ctr"] = pd.cut(fj_df["dist_ctr"], bins=bin_edges, labels=bin_labels)

    flank_profiles_all[sample_name]  = _profile(fj_df, "bin_bnd")
    center_profiles_all[sample_name] = _profile(fj_df, "bin_ctr")

    flanking = fj_df[fj_df["dist_bnd"] != 0]
    if len(flanking) > 2:
        rho, pval = spearmanr(flanking["dist_bnd"].abs(), flanking["methylation_level"])
    else:
        rho, pval = float("nan"), float("nan")

    spearman_results_all[sample_name] = {
        "rho":             rho,
        "pval":            pval,
        "n_g4s":           len(eg4_coords),
        "n_flanking_cpgs": len(flanking),
    }


In [ ]:
def add_rank(meth_df, reptime_df):
    meth_pd = meth_df.to_pandas() if isinstance(meth_df, pl.DataFrame) else meth_df
    return meth_pd.merge(
        reptime_df[["seqID", "start", "end", "rank"]],
        on=["seqID", "start", "end"],
        how="inner"
    )


In [ ]:
eG4s_reptime_meth = dict()
eG4s_rank_median = dict()
for sample in ["HG002", "CHM13"]:
    eG4s_reptime_meth[sample] = add_rank(meth_annotated["eG4", sample], reptime_dfs["eG4"])
    eG4s_rank_median[sample] = eG4s_reptime_meth[sample].groupby("rank")["avg_methylation"].median().reset_index()
eG4s_reptime_meth["HG002"]

In [ ]:
g4_df = pd.read_table(G4HUNTER)
regex_df = pd.read_table(QUADPARSER)

In [ ]:
for database in ["G4Hunter", "Quadparser"]:
    for sample_name in SAMPLES:
        
        # meth_annotated is pandas — use pandas syntax
        g4_with_cpg = (
            meth_annotated[(database, sample_name)]
            .dropna(subset=["avg_methylation"])
            [["seqID", "start", "end"]]
        )

        g4_cpg_pr = pr.PyRanges(
            g4_with_cpg.rename(columns={
                "seqID": "Chromosome",
                "start": "Start",
                "end":   "End"
            })
        )

        cpg_pr = pr.PyRanges(
            raw_meth[sample_name]
            .rename(columns={"seqID": "Chromosome",
                              "start": "Start",
                              "end":   "End"})
            [["Chromosome", "Start", "End", "methylation_level"]]
            .dropna(subset=["methylation_level"])
        )

        # find nearest G4 for each CpG within FLANK_SIZE
        nearest = cpg_pr.nearest(g4_cpg_pr, how=None)
        nearest_df = nearest.df.copy()
        nearest_df = nearest_df[nearest_df["Distance"] <= FLANK_SIZE]

        # CpGs inside G4s
        inside_df = cpg_pr.intersect(g4_cpg_pr).df.copy()
        inside_df["Distance"]     = 0
        inside_df["distance_bin"] = "inside"

        # bin by distance
        nearest_df["distance_bin"] = pd.cut(
            nearest_df["Distance"],
            bins=BINS,
            labels=[f"{BINS[i]}-{BINS[i+1]}bp" for i in range(len(BINS)-1)],
            include_lowest=True
        )

        # combine
        all_df = pd.concat([
            inside_df[["methylation_level", "Distance", "distance_bin"]],
            nearest_df[["methylation_level", "Distance", "distance_bin"]]
        ], ignore_index=True)

        # mean methylation per bin
        binned = (
            all_df.groupby("distance_bin", observed=True)["methylation_level"]
            .agg(["mean", "median", "count", "sem"])
            .reset_index()
        )

        # Spearman on distance vs methylation (excluding inside)
        rho, pval = spearmanr(
            nearest_df["Distance"],
            nearest_df["methylation_level"]
        )

        meth_distance_results[(database, sample_name)] = {
            "binned": binned,
            "all_df": all_df,
            "rho":    rho,
            "pval":   pval,
        }

        print(f"{database} | {sample_name}")
        print(f"  Spearman rho={rho:.3f}  p={pval:.2e}")
        print(binned.to_string())
        print()

In [ ]:
regions_df = pd.read_table(Path(os.getenv('WORK')).joinpath("compartments_coords.tsv.gz"))
regions_bed = BedTool.from_dataframe(regions_df).sort()
regions_df

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest
import pyranges as pr
import pandas as pd
import numpy as np

HYPO_THRESHOLD    = 0.2
TOTAL_COMPARISONS = len(["G4Hunter", "Quadparser"]) * len(SAMPLES)

def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

raw_meth = {
    "HG002": methylation_HG002_df,
    "CHM13": methylation_CHM13v2_df,
}

FIELDS = ["seqID", "start", "end"]
g4_df_id    = g4_df[FIELDS].rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"}).copy()
regex_df_id = regex_df[FIELDS].rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"}).copy()
g4_df_id["_g4_id"]    = range(len(g4_df_id))
regex_df_id["_g4_id"] = range(len(regex_df_id))

g4_prs = {
    "G4Hunter":   pr.PyRanges(g4_df_id),
    "Quadparser": pr.PyRanges(regex_df_id),
}
n_g4_total = {
    "G4Hunter":   len(g4_df_id),
    "Quadparser": len(regex_df_id),
}

meth_results = {}
for database in ["G4Hunter", "Quadparser"]:
    g4_pr = g4_prs[database]
    for sample_name in SAMPLES:
        meth_df = (
            raw_meth[sample_name]
            .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
            [["Chromosome", "Start", "End", "methylation_level"]]
            .dropna(subset=["methylation_level"])
        )
        cpg_pr       = pr.PyRanges(meth_df)
        n_meth_total = len(cpg_pr.df)

        g4_meth = cpg_pr.intersect(g4_pr             ).df["methylation_level"].dropna().values
        bg_meth = cpg_pr.intersect(g4_pr, invert=True).df["methylation_level"].dropna().values

        n_g4_with_meth    = g4_pr.overlap(cpg_pr).df["_g4_id"].nunique()
        frac_g4_with_meth = n_g4_with_meth / n_g4_total[database]
        n_meth_in_g4      = len(cpg_pr.intersect(g4_pr).df)
        frac_meth_in_g4   = n_meth_in_g4 / n_meth_total

        n, n_bg = len(g4_meth), len(bg_meth)
        stat_bg, pval_bg = mannwhitneyu(g4_meth, bg_meth, alternative="less")

        k_g4 = (g4_meth < HYPO_THRESHOLD).sum()
        k_bg = (bg_meth < HYPO_THRESHOLD).sum()
        frac_hypo_g4 = k_g4 / n   if n   > 0 else float("nan")
        frac_hypo_bg = k_bg / n_bg if n_bg > 0 else float("nan")
        _, pval_prop_raw = proportions_ztest([k_g4, k_bg], [n, n_bg])
        pval_prop        = min(pval_prop_raw * TOTAL_COMPARISONS, 1.0)
        h                = cohens_h(frac_hypo_g4, frac_hypo_bg)

        meth_results[(database, sample_name)] = {
            "g4_meth":            g4_meth,
            "bg_meth":            bg_meth,
            "pval_bg":            pval_bg,
            "r_rb_bg":            1 - (2 * stat_bg) / (n * n_bg),
            "n":                  n,
            "n_bg":               n_bg,
            "frac_g4_with_meth":  frac_g4_with_meth,
            "frac_meth_in_g4":    frac_meth_in_g4,
            "n_g4_with_meth":     n_g4_with_meth,
            "n_meth_in_g4":       n_meth_in_g4,
            "n_meth_total":       n_meth_total,
            "n_g4_total":         n_g4_total[database],
            "frac_hypo_g4":       frac_hypo_g4,
            "frac_hypo_bg":       frac_hypo_bg,
            "pval_prop":          pval_prop,
            "pval_prop_raw":      pval_prop_raw,
            "cohens_h":           h,
        }

In [ ]:
meth_results_random = {}
for database in ["G4Hunter", "Quadparser"]:
    g4_pr = g4_prs[database]
    for sample_name in SAMPLES:
        meth_df = (
            raw_meth[sample_name]
            .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
            [["Chromosome", "Start", "End", "methylation_level"]]
            .dropna(subset=["methylation_level"])
        )
        cpg_pr       = pr.PyRanges(meth_df)
        n_meth_total = len(cpg_pr.df)

        g4_meth = cpg_pr.intersect(g4_pr             ).df["methylation_level"].dropna().values
        bg_meth = cpg_pr.intersect(g4_pr, invert=True).df["methylation_level"].dropna().values

        # 1 random CpG per G4 for hypothesis testing
        joined = g4_pr.join(cpg_pr).df[["_g4_id", "methylation_level"]].dropna(subset=["methylation_level"])
        g4_meth_sampled = (
            joined.groupby("_g4_id", group_keys=False)
            .apply(lambda x: x.sample(1, random_state=42))
            ["methylation_level"].values
        )

        n_g4_with_meth    = g4_pr.overlap(cpg_pr).df["_g4_id"].nunique()
        frac_g4_with_meth = n_g4_with_meth / n_g4_total[database]
        n_meth_in_g4      = len(cpg_pr.intersect(g4_pr).df)
        frac_meth_in_g4   = n_meth_in_g4 / n_meth_total

        # --- background (shared by both approaches) ---
        n_bg  = len(bg_meth)
        k_bg  = (bg_meth < HYPO_THRESHOLD).sum()
        frac_hypo_bg = k_bg / n_bg if n_bg > 0 else float("nan")

        # --- sampled (1 CpG per G4) ---
        n_sampled        = len(g4_meth_sampled)
        stat_bg, pval_bg = mannwhitneyu(g4_meth_sampled, bg_meth, alternative="less")
        k_g4_sampled     = (g4_meth_sampled < HYPO_THRESHOLD).sum()
        frac_hypo_g4     = k_g4_sampled / n_sampled if n_sampled > 0 else float("nan")
        _, pval_prop_raw = proportions_ztest([k_g4_sampled, k_bg], [n_sampled, n_bg])
        pval_prop        = min(pval_prop_raw * TOTAL_COMPARISONS, 1.0)
        cohens_h_sampled = cohens_h(frac_hypo_g4, frac_hypo_bg)

        # --- all CpGs within G4s ---
        n_all        = len(g4_meth)
        k_g4_all     = (g4_meth < HYPO_THRESHOLD).sum()
        frac_hypo_g4_all = k_g4_all / n_all if n_all > 0 else float("nan")
        cohens_h_all     = cohens_h(frac_hypo_g4_all, frac_hypo_bg)

        meth_results_random[(database, sample_name)] = {
            # raw arrays
            "g4_meth":            g4_meth,
            "bg_meth":            bg_meth,
            "g4_meth_sampled":    g4_meth_sampled,
            # counts
            "n_all":              n_all,          # total CpGs in G4s
            "n_sampled":          n_sampled,      # 1-per-G4 sample size
            "n_bg":               n_bg,
            # hypothesis tests (sampled)
            "pval_bg":            pval_bg,
            "r_rb_bg":            1 - (2 * stat_bg) / (n_sampled * n_bg),
            "pval_prop":          pval_prop,
            "pval_prop_raw":      pval_prop_raw,
            # hypo fractions
            "frac_hypo_g4":       frac_hypo_g4,       # from sampled
            "frac_hypo_g4_all":   frac_hypo_g4_all,   # from all CpGs
            "frac_hypo_bg":       frac_hypo_bg,
            # effect sizes
            "cohens_h":           cohens_h_sampled,
            "cohens_h_all":       cohens_h_all,
            # descriptive
            "frac_g4_with_meth":  frac_g4_with_meth,
            "frac_meth_in_g4":    frac_meth_in_g4,
            "n_g4_with_meth":     n_g4_with_meth,
            "n_meth_in_g4":       n_meth_in_g4,
            "n_meth_total":       n_meth_total,
            "n_g4_total":         n_g4_total[database],
        }

meth_results_random["G4Hunter", "HG002"]

In [ ]:
from scipy.stats import mannwhitneyu
import pyranges as pr
import pandas as pd

FIELDS = ["seqID", "start", "end"]
g4_df_id    = g4_df[FIELDS].rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"}).copy()
regex_df_id = regex_df[FIELDS].rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"}).copy()
eg4_df_id   = eg4_df[FIELDS].rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"}).copy()
g4_df_id["_g4_id"]    = range(len(g4_df_id))
regex_df_id["_g4_id"] = range(len(regex_df_id))
eg4_df_id["_g4_id"]   = range(len(eg4_df_id))

g4_prs = {
    "G4Hunter":   pr.PyRanges(g4_df_id),
    "Quadparser": pr.PyRanges(regex_df_id),
    "eG4":        pr.PyRanges(eg4_df_id),
}
n_g4_total = {
    "G4Hunter":   len(g4_df_id),
    "Quadparser": len(regex_df_id),
    "eG4":        len(eg4_df_id),
}

RANKS = sorted(rep_df_merged["rank"].unique())
rep_prs = {
    rank: pr.PyRanges(
        rep_df_merged[rep_df_merged["rank"] == rank]
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
        [["Chromosome", "Start", "End"]]
    )
    for rank in RANKS
}

raw_meth = {
    "HG002": methylation_HG002_df,
    "CHM13": methylation_CHM13v2_df,
}

def _mwu(a, b, alt="less"):
    if len(a) > 0 and len(b) > 0:
        stat, pval = mannwhitneyu(a, b, alternative=alt)
        r_rb = 1 - (2 * stat) / (len(a) * len(b))
        return pval, r_rb
    return float("nan"), float("nan")

rep_meth_results = {}

for sample_name in SAMPLES:
    meth_df = (
        raw_meth[sample_name]
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
        [["Chromosome", "Start", "End", "methylation_level"]]
        .dropna(subset=["methylation_level"])
    )
    full_cpg_pr = pr.PyRanges(meth_df)

    for rank in RANKS:
        rep_pr       = rep_prs[rank]
        cpg_pr       = full_cpg_pr.intersect(rep_pr)   # CpGs in this decile — computed once
        n_meth_total = len(cpg_pr.df)

        g4_meth_arrays = {}  # store per-database methylation arrays for cross-comparisons

        for database in ["G4Hunter", "Quadparser", "eG4"]:
            g4_pr          = g4_prs[database]
            g4_in_decile   = g4_pr.overlap(rep_pr)
            n_g4_in_decile = g4_in_decile.df["_g4_id"].nunique()

            g4_meth = cpg_pr.intersect(g4_pr             ).df["methylation_level"].dropna().values
            bg_meth = cpg_pr.intersect(g4_pr, invert=True).df["methylation_level"].dropna().values
            g4_meth_arrays[database] = g4_meth

            n_g4_with_meth    = g4_in_decile.overlap(cpg_pr).df["_g4_id"].nunique()
            frac_g4_with_meth = n_g4_with_meth / n_g4_in_decile if n_g4_in_decile > 0 else float("nan")
            n_meth_in_g4      = len(cpg_pr.intersect(g4_pr).df)
            frac_meth_in_g4   = n_meth_in_g4 / n_meth_total if n_meth_total > 0 else float("nan")

            pval_bg, r_rb_bg = _mwu(g4_meth, bg_meth)

            rep_meth_results[(database, sample_name, rank)] = {
                "g4_meth":            g4_meth,
                "bg_meth":            bg_meth,
                "pval_bg":            pval_bg,
                "r_rb_bg":            r_rb_bg,
                "n":                  len(g4_meth),
                "n_bg":               len(bg_meth),
                "frac_g4_with_meth":  frac_g4_with_meth,
                "frac_meth_in_g4":    frac_meth_in_g4,
                "n_g4_with_meth":     n_g4_with_meth,
                "n_g4_in_decile":     n_g4_in_decile,
                "n_meth_in_g4":       n_meth_in_g4,
                "n_meth_total":       n_meth_total,
            }

        # eG4 vs G4Hunter and eG4 vs Quadparser per decile
        for other_db in ["G4Hunter", "Quadparser"]:
            eg4_meth   = g4_meth_arrays["eG4"]
            other_meth = g4_meth_arrays[other_db]
            pval, r_rb = _mwu(eg4_meth, other_meth)

            rep_meth_results[(f"eG4_vs_{other_db}", sample_name, rank)] = {
                "eg4_meth":   eg4_meth,
                "other_meth": other_meth,
                "pval":       pval,
                "r_rb":       r_rb,
                "n_eg4":      len(eg4_meth),
                "n_other":    len(other_meth),
            }


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pyranges as pr

HYPO_THRESHOLD = 0.2
DB_PALETTE     = {"G4Hunter": "#ba8de0", "Quadparser": "#6dbf9e", "eG4": "#f28c5e"}

# ── 1. Per-G4 mean methylation → hypomethylation fraction per decile ──────────
hypo_g4_results = {}

for sample_name in SAMPLES:
    meth_df = (
        raw_meth[sample_name]
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
        [["Chromosome", "Start", "End", "methylation_level"]]
        .dropna(subset=["methylation_level"])
    )
    full_cpg_pr = pr.PyRanges(meth_df)

    for rank in RANKS:
        rep_pr = rep_prs[rank]
        cpg_pr = full_cpg_pr.intersect(rep_pr)   # CpGs restricted to this decile

        for database in ["G4Hunter", "Quadparser", "eG4"]:
            g4_pr        = g4_prs[database]
            g4_in_decile = g4_pr.overlap(rep_pr)

            if len(g4_in_decile.df) == 0:
                hypo_g4_results[(database, sample_name, rank)] = {
                    "frac_hypo": float("nan"), "n_g4_with_meth": 0, "n_hypo": 0,
                }
                continue

            joined = (
                g4_in_decile.join(cpg_pr).df
                [["_g4_id", "methylation_level"]]
                .dropna(subset=["methylation_level"])
            )

            if len(joined) == 0:
                hypo_g4_results[(database, sample_name, rank)] = {
                    "frac_hypo": float("nan"), "n_g4_with_meth": 0, "n_hypo": 0,
                }
                continue

            per_g4_mean    = joined.groupby("_g4_id")["methylation_level"].mean()
            n_g4_with_meth = len(per_g4_mean)
            n_hypo         = (per_g4_mean < HYPO_THRESHOLD).sum()

            hypo_g4_results[(database, sample_name, rank)] = {
                "frac_hypo":      n_hypo / n_g4_with_meth,
                "n_g4_with_meth": n_g4_with_meth,
                "n_hypo":         int(n_hypo),
                "per_g4_mean":    per_g4_mean.values,
            }


# ── 2. Barplot ─────────────────────────────────────────────────────────────────
DATABASES  = ["G4Hunter", "Quadparser", "eG4"]
x          = np.arange(len(RANKS))
bar_width  = 0.25
offsets    = np.array([-1, 0, 1]) * bar_width

fig, axes = plt.subplots(1, len(SAMPLES), figsize=(14, 5), sharey=True)

for ax, sample_name in zip(axes, SAMPLES):
    for db, offset in zip(DATABASES, offsets):
        fracs = [hypo_g4_results[(db, sample_name, r)]["frac_hypo"] for r in RANKS]
        ax.bar(x + offset, fracs, width=bar_width,
               color=DB_PALETTE[db], label=db, edgecolor="black", linewidth=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels([str(r) for r in RANKS], fontsize=14)
    ax.tick_params(axis="y", labelsize=14)
    ax.set_xlabel("Replication timing decile", fontsize=16)
    ax.set_ylabel("Fraction hypomethylated G4s" if ax is axes[0] else "", fontsize=16)
    ax.text(0.5, 1.02, sample_name, transform=ax.transAxes,
            ha="center", fontsize=16,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="lightgray",
                      edgecolor="black", linewidth=1.5))
    ax.grid(axis="y", lw=0.4, alpha=0.6)
    ax.set_ylim(0, 1)
    if ax is axes[-1]:
        ax.legend(fontsize=13, frameon=True, fancybox=True)

fig.savefig(target_fig / "frac_hypo_g4s_rep_deciles.pdf", transparent=True, bbox_inches="tight")
fig.savefig(target_fig / "frac_hypo_g4s_rep_deciles.png", dpi=400, transparent=True, bbox_inches="tight")
plt.show()


In [ ]:
rep_meth_results["G4Hunter", "HG002", 5]

In [ ]:
methylation_HG002_bed = BedTool.from_dataframe(
    methylation_HG002_df).sort()
g4_bed = BedTool.from_dataframe(
    g4_df
).sort()
g4_bed.intersect(methylation_HG002_bed, F=1.0, u=True).count()

In [ ]:
meth_results["G4Hunter", "HG002"]

In [ ]:
meth_results["Quadparser", "HG002"]

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import patches as mpatches

SAMPLES = ["HG002", "CHM13"]
palette = {"Background": "#a5acb0", "G4": "#ba8de0"}

for database in ["G4Hunter", "Quadparser"]:
    fig = plt.figure(figsize=(10, 7))
    gs  = fig.add_gridspec(2, len(SAMPLES), height_ratios=[1, 2.5], hspace=0.1, wspace=0.15)

    axes_kde = [fig.add_subplot(gs[0, i]) for i in range(len(SAMPLES))]
    axes_vln = [fig.add_subplot(gs[1, i]) for i in range(len(SAMPLES))]

    for i, (ax_kde, ax_vln, sample_name) in enumerate(zip(axes_kde, axes_vln, SAMPLES)):
        res     = meth_results_random[(database, sample_name)]
        g4_meth = res["g4_meth"]
        bg_meth = res["bg_meth"]
        r_rb    = res["cohens_h"]       # effect size from 1-per-G4 sample
        n       = res["n_all"]          # total CpGs in G4s for reporting
        frac_g4 = res["frac_g4_with_meth"]

        plot_data = pd.concat([
            pd.DataFrame({"methylation": g4_meth, "group": "G4"}),
            pd.DataFrame({"methylation": bg_meth,  "group": "Background"}),
        ], ignore_index=True)

        # ── KDE panel ─────────────────────────────────────────────────────────
        sns.kdeplot(bg_meth, ax=ax_kde, color=palette["Background"],
                    fill=True, alpha=0.4, linewidth=1.5, label="Background")
        sns.kdeplot(g4_meth, ax=ax_kde, color=palette["G4"],
                    fill=True, alpha=0.4, linewidth=1.5, label="G4")
        ax_kde.set_xlabel("")
        ax_kde.set_ylabel("Density" if i == 0 else "", fontsize=16)
        ax_kde.set_yticks([])
        ax_kde.tick_params(labelsize=14)
        ax_kde.set_title(sample_name, fontsize=18, pad=8,
                         bbox=dict(boxstyle="round,pad=0.4", facecolor="lightgray",
                                   edgecolor="black", linewidth=1.5))
        for spine in ax_kde.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)
        ax_kde.spines["left"].set_visible(False)
        ax_kde.legend(handles=[], frameon=False)

        # ── Violin panel ───────────────────────────────────────────────────────
        sns.violinplot(
            data=plot_data, x="group", y="methylation",
            order=["Background", "G4"], palette=palette,
            inner="quartile", cut=0, linewidth=1.5, ax=ax_vln,
        )
        ax_vln.set_xlabel("")
        ax_vln.set_ylabel("Methylation" if i == 0 else "", fontsize=17)
        ax_vln.tick_params(labelsize=17)
        ax_vln.text(0.5, -0.15, f"$h$ = {r_rb:.3f}  (n = {n:,})",
                    transform=ax_vln.transAxes,
                    ha="center",
                    fontsize=12,
                    style="italic")
        ax_vln.grid(axis="y", lw=0.4, alpha=0.6)
        sns.despine(ax=ax_vln)

    fig.savefig(target_fig / f"g4_meth_vs_background_{database}.r.pdf",
                transparent=True, bbox_inches="tight")
    fig.savefig(target_fig / f"g4_meth_vs_background_{database}.r.png",
                dpi=400, transparent=True, bbox_inches="tight")
    plt.show()


In [ ]:
HYPO_T  = 0.2
HYPER_T = 0.8

for database in ["G4Hunter", "Quadparser"]:
    for sample_name in SAMPLES:
        res = meth_results_random[(database, sample_name)]
        g4_meth = res["g4_meth"]

        n_g4_total     = res["n_g4_total"]
        n_g4_with_meth = res["n_g4_with_meth"]
        frac_with_meth = n_g4_with_meth / n_g4_total

        # ── per-CpG breakdown ─────────────────────────────────────────────────
        n_cpg_total = len(g4_meth)
        n_hypo_cpg  = (g4_meth < HYPO_T).sum()
        n_meth_cpg  = ((g4_meth >= HYPO_T) & (g4_meth <= HYPER_T)).sum()
        n_hyper_cpg = (g4_meth > HYPER_T).sum()

        # ── per-G4 mean breakdown ─────────────────────────────────────────────
        joined = g4_prs[database].join(
            pr.PyRanges(
                raw_meth[sample_name]
                .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
                [["Chromosome", "Start", "End", "methylation_level"]]
                .dropna(subset=["methylation_level"])
            )
        ).df[["_g4_id", "methylation_level"]].dropna(subset=["methylation_level"])

        per_g4_mean   = joined.groupby("_g4_id")["methylation_level"].mean()
        n_g4_measured = len(per_g4_mean)
        n_hypo_g4     = (per_g4_mean < HYPO_T).sum()
        n_meth_g4     = ((per_g4_mean >= HYPO_T) & (per_g4_mean <= HYPER_T)).sum()
        n_hyper_g4    = (per_g4_mean > HYPER_T).sum()

        print(f"\n{'='*60}")
        print(f"  {database}  |  {sample_name}")
        print(f"{'='*60}")
        print(f"  Total G4s                : {n_g4_total:>12,}")
        print(f"  G4s with ≥1 CpG          : {n_g4_with_meth:>12,}  ({frac_with_meth*100:.2f}%)")
        print()
        print(f"  --- Per-CpG (all CpGs within G4s) ---")
        print(f"  Total CpGs in G4s        : {n_cpg_total:>12,}")
        print(f"  Hypomethylated (<{HYPO_T})   : {n_hypo_cpg:>12,}  ({n_hypo_cpg/n_cpg_total*100:.1f}%)")
        print(f"  Methylated ({HYPO_T}–{HYPER_T})    : {n_meth_cpg:>12,}  ({n_meth_cpg/n_cpg_total*100:.1f}%)")
        print(f"  Hypermethylated (>{HYPER_T})  : {n_hyper_cpg:>12,}  ({n_hyper_cpg/n_cpg_total*100:.1f}%)")
        print(f"  Sum check                : {n_hypo_cpg+n_meth_cpg+n_hyper_cpg:>12,}")
        print()
        print(f"  --- Per-G4 mean methylation ---")
        print(f"  G4s with mean computed   : {n_g4_measured:>12,}")
        print(f"  Hypomethylated (<{HYPO_T})   : {n_hypo_g4:>12,}  ({n_hypo_g4/n_g4_measured*100:.1f}%)")
        print(f"  Methylated ({HYPO_T}–{HYPER_T})    : {n_meth_g4:>12,}  ({n_meth_g4/n_g4_measured*100:.1f}%)")
        print(f"  Hypermethylated (>{HYPER_T})  : {n_hyper_g4:>12,}  ({n_hyper_g4/n_g4_measured*100:.1f}%)")
        print(f"  Sum check                : {n_hypo_g4+n_meth_g4+n_hyper_g4:>12,}")


In [ ]:
import pyranges as pr

def assign_eg4_flag(matched_df, eg4_df, min_frac=0.5):
    to_pr = lambda df: pr.PyRanges(
        (df.to_pandas() if isinstance(df, pl.DataFrame) else df)
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
    )

    joined = to_pr(matched_df).join(to_pr(eg4_df), suffix="_eg4").df

    joined["overlap"] = (
        joined[["End", "End_eg4"]].min(axis=1) -
        joined[["Start", "Start_eg4"]].max(axis=1)
    ).clip(lower=0)
    joined["frac"] = joined["overlap"] / (joined["End"] - joined["Start"])

    hits = (
        joined[joined["frac"] >= min_frac][["Chromosome", "Start", "End"]]
        .drop_duplicates()
        .rename(columns={"Chromosome": "seqID", "Start": "start", "End": "end"})
        .assign(eG4=1)
    )

    return (
        matched_df
        .merge(hits[["seqID", "start", "end", "eG4"]],
               on=["seqID", "start", "end"], how="left")
        .assign(eG4=lambda d: d["eG4"].fillna(0).astype(int))
    )

In [ ]:
target_fig / f"rank_biserial_{ALGO}_{TIER}.png"

## Methylation & Conservation

In [ ]:
haplotypes = pd.read_table("https://raw.githubusercontent.com/human-pangenomics/HPP_Year1_Data_Freeze_v1.0/refs/heads/main/sample_metadata/hprc_year1_sample_metadata.txt")
ancestries = dict(zip(haplotypes["Sample"], 
                      haplotypes["Superpopulation"]))
len(ancestries)
haplotypes

In [ ]:
g4_df = pd.read_table(G4HUNTER)
g4_df

In [ ]:
import polars as pl
path = "/scratch/10904/nikolchanchan/MAFin_results_CHM13_g4/CHM13_pG4s_CHM13.g4hunter.unique_pG4s_CHM13.g4hunter.unique_motif_hits.csv.gz"
cols = pd.read_csv(path, nrows=0).columns.tolist()[:6]
conservation_df = pl.read_csv(path, separator=",")
conservation_df

In [ ]:
import pyranges as pr

haplotypes_in_MAFin = list(filter(lambda x: "#" in x, conservation_df.columns[5:]))
len(haplotypes_in_MAFin)
conservation_df = conservation_df.with_columns(
    total_non_null=pl.sum_horizontal(pl.col(haplotype).is_not_null() for haplotype in haplotypes_in_MAFin if haplotype.split("#")[0] in ancestries)
)
filtered_conservation_df = conservation_df.filter(pl.col("total_non_null") >= 44)
filtered_conservation_df = filtered_conservation_df.with_columns(
    pl.col("motif_hit_info").str.extract(r"(.+):(\d+)-(\d+),(.+)", group_index=1).alias("seqID"),
    pl.col("motif_hit_info").str.extract(r"(.+):(\d+)-(\d+),(.+)", group_index=2).cast(pl.Int64).alias("start"),
    pl.col("motif_hit_info").str.extract(r"(.+):(\d+)-(\d+),(.+)", group_index=3).cast(pl.Int64).alias("end"),
    pl.col("motif_hit_info").str.extract(r"(.+):(\d+)-(\d+),(.+)", group_index=4).alias("sequence"),
).with_columns(
    (pl.col("end") + 1).alias("end")
)

cons_pr = pr.PyRanges(filtered_conservation_df.select(["seqID", "start", "end", "score"]).to_pandas()
                            .rename(columns={
                                "seqID": "Chromosome", 
                                "start": "Start", 
                                "end": "End"})
                                )

In [ ]:
filtered_conservation_df.head()

In [ ]:
meth_datasets = {
                  ("CHM13", "G4Hunter"): (meth_annotated["G4Hunter", "CHM13"], filtered_conservation_df),
                   ("HG002", "G4Hunter"): (meth_annotated["G4Hunter", "HG002"], filtered_conservation_df),
                 
                   }

meth_cons_data = {}
for group, (df, cons_df) in meth_datasets.items():
    if isinstance(df, pl.DataFrame):
        df = df.to_pandas()
    g4hunter_df_HG002_meth_cons = (
                            df.query("seqID != 'chrY'")
                              .merge(
                                    cons_df
                                        .select(["seqID", "start", "end", "score"])
                                        .to_pandas(), 

                                            left_on=["seqID", "start", "end"],
                                            right_on=["seqID", "start", "end"],
                                            how="inner",  
                                            suffixes=("", "_g4")
                                                    
                                )
    )
    
    # g4hunter_df_HG002_meth_cons.dropna(subset=["score"], inplace=True)
    # g4hunter_df_HG002_meth_cons.loc[:, "Conservation"] = (g4hunter_df_HG002_meth_cons["score"] > 97).astype(int)
    meth_cons_data[group] = pl.from_pandas(g4hunter_df_HG002_meth_cons)\
                  .with_columns(
                        pl.when(pl.col("score") > 98)
                        .then(pl.lit("Conserved"))
                        .otherwise(pl.lit("Non-conserved"))
                        .alias("Conservation")
                    )
    
    # g4hunter_df_HG002_meth_cons.loc[:, "score"] = g4hunter_df_HG002_meth_cons["score"].fillna(0.0)
    # g4hunter_df_HG002_meth_cons

In [ ]:
meth_cons_data["HG002", method]["Conservation"].value_counts()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

HYPO_THRESHOLD = 0.2
method  = "G4Hunter"
SAMPLES = ["HG002", "CHM13"]
palette_cons = {"Conserved": "#86a3c8", "Non-conserved": "#ff1cb3"}

def _stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

fig = plt.figure(figsize=(4 * len(SAMPLES), 7))
gs  = fig.add_gridspec(2, len(SAMPLES), height_ratios=[1, 2.5], hspace=0.1, wspace=0.15)
axes_kde = [fig.add_subplot(gs[0, i]) for i in range(len(SAMPLES))]
axes_vln = [fig.add_subplot(gs[1, i]) for i in range(len(SAMPLES))]

for i, (ax_kde, ax_vln, sample_name) in enumerate(zip(axes_kde, axes_vln, SAMPLES)):
    df = (meth_cons_data[sample_name, method]
          .filter(pl.col("seqID") != "chrY")
          .to_pandas())

    cons_meth    = df[df["Conservation"] == "Conserved"]["avg_methylation"].dropna()
    noncons_meth = df[df["Conservation"] == "Non-conserved"]["avg_methylation"].dropna()

    # hypomethylated fractions
    k_cons,    n_cons    = (cons_meth    < HYPO_THRESHOLD).sum(), len(cons_meth)
    k_noncons, n_noncons = (noncons_meth < HYPO_THRESHOLD).sum(), len(noncons_meth)
    frac_cons    = k_cons    / n_cons
    frac_noncons = k_noncons / n_noncons

    _, pval = proportions_ztest([k_cons, k_noncons], [n_cons, n_noncons])

    # ── KDE ────────────────────────────────────────────────────────────────────
    sns.kdeplot(cons_meth.values,    ax=ax_kde, color=palette_cons["Conserved"],
                fill=True, alpha=0.4, linewidth=1.5)
    sns.kdeplot(noncons_meth.values, ax=ax_kde, color=palette_cons["Non-conserved"],
                fill=True, alpha=0.4, linewidth=1.5)
    ax_kde.axvline(HYPO_THRESHOLD, color="black", lw=0.8, ls="--", alpha=0.5)
    ax_kde.set_xlabel("")
    ax_kde.set_ylabel("Density" if i == 0 else "", fontsize=16)
    ax_kde.set_yticks([])
    ax_kde.tick_params(labelsize=14)
    ax_kde.set_title(sample_name, fontsize=18, pad=8,
                     bbox=dict(boxstyle="round,pad=0.4", facecolor="lightgray",
                               edgecolor="black", linewidth=1.5))
    for spine in ax_kde.spines.values():
        spine.set_visible(True); spine.set_linewidth(0.8)
    ax_kde.spines["left"].set_visible(False)
    ax_kde.legend(handles=[], frameon=False)

    # two pies: conserved (left) | non-conserved (right)
    for frac, color, x_pos in [
        (frac_cons,    palette_cons["Conserved"],     0.22),
        (frac_noncons, palette_cons["Non-conserved"], 0.58),
    ]:
        ax_pie = ax_kde.inset_axes([x_pos, 0.37, 0.14, 0.72])
        ax_pie.pie([frac, 1 - frac], colors=[color, "#e0e0e0"],
                   startangle=90, wedgeprops=dict(linewidth=1.5, edgecolor="black"))

    # significance above KDE
    # replace the ax_kde.text significance line with this:
    ax_kde.text(0.5, 0.97, _stars(pval), transform=ax_kde.transAxes,
                ha="center", va="top", fontsize=16, color="black")
    import numpy as np

    def cohens_h(p1, p2):
        return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

    h = cohens_h(frac_cons, frac_noncons)
    print(h, frac_cons, frac_noncons, pval)


    # ── Violin ─────────────────────────────────────────────────────────────────
    plot_data = pd.concat([
        pd.DataFrame({"methylation": cons_meth.values,    "group": "Conserved"}),
        pd.DataFrame({"methylation": noncons_meth.values, "group": "Non-conserved"}),
    ], ignore_index=True)

    sns.violinplot(
        data=plot_data, x="group", y="methylation",
        order=["Conserved", "Non-conserved"], palette=palette_cons,
        inner="quartile", cut=0, linewidth=1.5, ax=ax_vln,
    )
    ax_vln.axhline(HYPO_THRESHOLD, color="black", lw=0.8, ls="--", alpha=0.5)
    ax_vln.set_xlabel("")
    ax_vln.set_ylabel("Methylation" if i == 0 else "", fontsize=17)
    ax_vln.tick_params(labelsize=15)
    ax_vln.grid(axis="y", lw=0.4, alpha=0.6)
    sns.despine(ax=ax_vln)

fig.savefig(f"{target_fig}/methylation_conserved_HG002_CHM13v2.pdf",
            transparent=True, bbox_inches="tight")
fig.savefig(f"{target_fig}/methylation_conserved_HG002_CHM13v2.png",
            dpi=400, transparent=True, bbox_inches="tight")
plt.show()


In [ ]:
import pyranges as pr
import polars as pl

rep_df = pd.read_csv(
                f"{os.getenv('SCRATCH')}/g4_t2t_revisions_data/bg02es_replitime.deciles.hs1.bed",
                header=None,
                sep="\t",
                names=["seqID", "start", "end", "decile", "rank"]
             ).sort_values(["seqID", "start"]).reset_index(drop=True)
rep_df["rank"] = rep_df["rank"].map({i: 11 - i for i in range(1, 11)})

def merge_overlapping(df: pd.DataFrame, col: str) -> pd.DataFrame:
    collection = set(df[col])
    df_collection = []
    for c in tqdm(collection):
        df_temp = pd.read_csv(
                        BedTool.from_dataframe(
                                        df[df[col] == c]
                        ).sort().merge(c="3", o="count").sort().fn,
                    header=None,
                    sep="\t",
                    names=["seqID", "start", "end", "counts"]
        )
        df_temp[col] = c
        df_collection.append(df_temp)
    return pd.concat(df_collection, ignore_index=True)


print(rep_df.shape)
rep_df_merged = merge_overlapping(rep_df, col="rank")
rep_df_merged

rep_pr = pr.PyRanges(
    rep_df_merged.rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
)

def assign_rep_time(joint_df):
    g4_pr = pr.PyRanges(
        joint_df.rename({"seqID": "Chromosome", "start": "Start", "end": "End"}).to_pandas()
    )
    joined = g4_pr.join(rep_pr, suffix="_rep")
    df = pl.from_pandas(joined.df)
    return (
        df.filter(
            (pl.col("Start_rep") <= pl.col("Start")) &
            (pl.col("End_rep")   >= pl.col("End"))
        )
        .rename({"Chromosome": "seqID", "Start": "start", "End": "end"})
    )


g4_cons_reptime = {}
for group, df in meth_cons_data.items():
    g4_cons_reptime[group] = assign_rep_time(df)
    print(f"{group}: {len(g4_cons_reptime[group]):,} rows")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from statannotations.Annotator import Annotator
from scipy.stats import mannwhitneyu
from pathlib import Path

midgreen     = "#86a3c8"
sample_order = ["HG002", "CHM13"]
cons_order   = ["Conserved", "Non-conserved"]
palette      = {"Conserved": midgreen, "Non-conserved": "#f7318e"}

hg_color  = "#1f77b4"
chm_color = "#aec7e8"
method = "G4Hunter"

def add_sample(df, name):
    if "sample" in df.columns:
        df = df.drop("sample")
    return df.with_columns(pl.lit(name).alias("sample"))

plot_df = (
    pl.concat([
        add_sample(meth_cons_data["HG002", method], "HG002"),
        add_sample(meth_cons_data["CHM13", method], "CHM13"),
    ], how="diagonal")
    .filter(pl.col("seqID") != "chrY")
    .to_pandas()
)
def get_reptime_deltas(df):
    df = df.filter(pl.col("seqID") != "chrY").to_pandas()
    df["rank"] = df["rank"].astype(int)
    rank_order = sorted(df["rank"].unique().tolist())
    deltas, pvals, sizes = [], [], []
    for r in rank_order:
        sub = df[df["rank"] == r]
        x = sub[sub["Conservation"] == "Conserved"]["avg_methylation"].dropna().values
        y = sub[sub["Conservation"] == "Non-conserved"]["avg_methylation"].dropna().values
        sizes.append((len(x), len(y)))
        if len(x) < 2 or len(y) < 2:
            deltas.append(0.0)
            pvals.append(float("nan"))
            continue
        U1, p = mannwhitneyu(x, y, alternative="two-sided")
        deltas.append((2 * U1) / (len(x) * len(y)) - 1)
        pvals.append(p)
    return rank_order, deltas, pvals, sizes


rank_order_hg,  deltas_hg,  pvals_hg,  sizes_hg  = get_reptime_deltas(g4_cons_reptime["HG002", method])
rank_order_chm, deltas_chm, pvals_chm, sizes_chm = get_reptime_deltas(g4_cons_reptime["CHM13", method])
print(f"{'Rank':>4} | {'HG002 δ':>8} | {'p':>10} | {'n1':>6} | {'n2':>6} | {'CHM13 δ':>8} | {'p':>10} | {'n1':>6} | {'n2':>6}")
print("-" * 80)
for r, d_hg, p_hg, (n1_hg, n2_hg), d_chm, p_chm, (n1_chm, n2_chm) in zip(
    rank_order_hg, deltas_hg, pvals_hg, sizes_hg,
    deltas_chm, pvals_chm, sizes_chm
):
    print(f"{r:>4} | {d_hg:>+8.4f} | {p_hg:>10.3e} | {n1_hg:>6,} | {n2_hg:>6,} | {d_chm:>+8.4f} | {p_chm:>10.3e} | {n1_chm:>6,} | {n2_chm:>6,}")

fig = plt.figure(figsize=(13, 7))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1.2, 2], wspace=0.04)
ax_bar = fig.add_subplot(gs[0])
ax_box = fig.add_subplot(gs[1])
bar_h = 0.35
n = len(rank_order_hg)

for i, (d_hg, d_chm) in enumerate(zip(deltas_hg, deltas_chm)):
    ax_bar.barh(i + bar_h / 2, d_hg,  height=bar_h, color=hg_color,
                edgecolor="black", linewidth=0.7, label="HG002" if i == 0 else "")
    ax_bar.barh(i - bar_h / 2, d_chm, height=bar_h, color=chm_color,
                edgecolor="black", linewidth=0.7, label="CHM13" if i == 0 else "")

ax_bar.axvline(0, color="black", lw=1.0)
ax_bar.set_yticks(range(n))
ax_bar.set_yticklabels([str(r) for r in rank_order_hg], fontsize=17)
ax_bar.set_ylabel("Replication Timing", fontsize=20)
ax_bar.set_xlabel("Cliff's δ", fontsize=18)
ax_bar.tick_params(axis="x", labelsize=17)
ax_bar.tick_params(axis="y", labelsize=17)
ax_bar.grid(lw=0.3, alpha=0.5, axis="x", zorder=0)
ax_bar.set_axisbelow(True)
ax_bar.set_ylim(-0.5, n - 0.5)
ax_bar.legend(fontsize=14, frameon=True, loc="upper left")

# dotted border only — no internal dots
for spine in ax_bar.spines.values():
    spine.set_linestyle((0, (4, 4)))
    spine.set_linewidth(1.2)
    spine.set_edgecolor("gray")

# --- boxplot (y-axis on right) ---
sns.boxplot(
    data=plot_df, x="sample", y="avg_methylation",
    hue="Conservation", order=sample_order, hue_order=cons_order,
    palette=palette, width=0.5,
    boxprops=dict(linewidth=2, edgecolor="black"),
    medianprops=dict(color="black", linewidth=2.5),
    showmeans=True,
    meanprops=dict(marker="^", markersize=13, markerfacecolor="white",
                   markeredgecolor="black", markeredgewidth=1.5),
    ax=ax_box,
)
pairs = [
    (("HG002", "Conserved"), ("HG002", "Non-conserved")),
    (("CHM13", "Conserved"), ("CHM13", "Non-conserved")),
]
annotator = Annotator(ax_box, pairs, data=plot_df,
                      x="sample", y="avg_methylation",
                      hue="Conservation", order=sample_order, hue_order=cons_order)
annotator.configure(test="Mann-Whitney", comparisons_correction="BH",
                    text_format="star", fontsize=18, loc="outside", verbose=False)
annotator.apply_and_annotate()

ax_box.axhline(0.5, linestyle="--", color="crimson", lw=2.0)
ax_box.grid(lw=0.4, alpha=0.6, zorder=0)
ax_box.set_axisbelow(True)
ax_box.set_xlabel("")
ax_box.set_ylabel("Methylation", fontsize=20)
ax_box.tick_params(axis="x", labelsize=20)
ax_box.tick_params(axis="y", labelsize=17)
ax_box.yaxis.tick_right()
ax_box.yaxis.set_label_position("right")
ax_box.spines["left"].set_visible(False)
ax_box.spines["right"].set_visible(True)
ax_box.legend(fontsize=14, 
            frameon=True, 
            fancybox=True, 
            shadow=True, 
            bbox_to_anchor=(-0.15, 1.15), 
            loc="upper left")

figures = Path(f"{os.getenv('SCRATCH')}/figures_g4_t2t/methylation")
figures.mkdir(exist_ok=True, parents=True)
fig.savefig(figures / f"methylation_retained_HG002_CHM13v2_{method}_with_rep_time.png",
            transparent=True, 
            format="png", 
            dpi=300, 
            bbox_inches="tight")
plt.show()

In [ ]:
regions_df = pd.read_table(Path(os.getenv('WORK')).joinpath("compartments_coords.tsv.gz"))
regions_bed = BedTool.from_dataframe(regions_df).sort()

In [ ]:
import pyranges as pr
import polars as pl
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Prepare regions PyRanges (4 cols)
regions_pr = pr.PyRanges(
    regions_df.rename(columns={"seqID": "Chromosome", 
                    "start": "Start", 
                    "end": "End"})
)

def map_to_regions(joint_df):
    g4_pr = pr.PyRanges(
        joint_df.rename({"seqID": "Chromosome", "start": "Start", "end": "End"}).to_pandas()
    )
    joined = g4_pr.join(regions_pr, suffix="_reg")
    df = pl.from_pandas(joined.df)
    # f=1.0: G4 fully contained within region
    return (
        df.filter(
            (pl.col("Start_reg") <= pl.col("Start")) &
            (pl.col("End_reg")   >= pl.col("End"))
        )
        .rename({"Chromosome": "seqID", "Start": "start", "End": "end"})
    )

methylation_g4_CHM13meth_df = map_to_regions(meth_cons_data["CHM13", "G4Hunter"])
methylation_g4_HG002meth_df = map_to_regions(meth_cons_data["HG002", "G4Hunter"])

In [ ]:
meth_cons_data["CHM13", "G4Hunter"]

In [ ]:
from scipy.stats import mannwhitneyu, ks_2samp
from statsmodels.stats.proportion import proportions_ztest
import numpy as np
import pandas as pd

HYPO_THRESHOLD = 0.2

def sig_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

cat = "avg_methylation"
total_comparisons = len(set(methylation_g4_HG002meth_df["group"]))
compartments_all  = methylation_g4_HG002meth_df["group"].unique().to_list()

rows = []
for compartment in sorted(compartments_all):
    temp = methylation_g4_HG002meth_df.filter(pl.col("group") == compartment)
    g1 = temp.filter(pl.col("Conservation") == "Conserved")[cat].drop_nulls().to_numpy()
    g2 = temp.filter(pl.col("Conservation") == "Non-conserved")[cat].drop_nulls().to_numpy()
    if len(g1) < 2 or len(g2) < 2:
        continue

    u_stat, mw_p = mannwhitneyu(g1, g2, alternative="two-sided")
    delta = (2 * u_stat) / (len(g1) * len(g2)) - 1
    ks_stat, ks_p = ks_2samp(g1, g2, alternative="two-sided")

    # proportions test on fraction hypomethylated
    k1, n1 = (g1 < HYPO_THRESHOLD).sum(), len(g1)
    k2, n2 = (g2 < HYPO_THRESHOLD).sum(), len(g2)
    frac_cons    = k1 / n1
    frac_noncons = k2 / n2
    _, prop_p = proportions_ztest([k1, k2], [n1, n2])
    h = cohens_h(frac_cons, frac_noncons)

    rows.append({
        "compartment":    compartment,
        "n_conserved":    n1,
        "n_nonconserved": n2,
        "cliffs_delta":   delta,
        "mw_p_bonf":      min(mw_p   * total_comparisons, 1.0),
        "ks_stat":        round(ks_stat, 4),
        "ks_p_bonf":      min(ks_p   * total_comparisons, 1.0),
        "frac_hypo_cons":    frac_cons,
        "frac_hypo_noncons": frac_noncons,
        "cohens_h":          h,
        "prop_p_bonf":       min(prop_p * total_comparisons, 1.0),
    })

results_df = pd.DataFrame(rows).sort_values("cliffs_delta", ascending=False).reset_index(drop=True)
results_df["mw_sig"]   = results_df["mw_p_bonf"].apply(sig_stars)
results_df["ks_sig"]   = results_df["ks_p_bonf"].apply(sig_stars)
results_df["prop_sig"] = results_df["prop_p_bonf"].apply(sig_stars)
pd.options.display.float_format = "{:,.4f}".format
results_df


In [ ]:
from scipy.stats import mannwhitneyu, ks_2samp
import numpy as np
import seaborn as sns

darkgreen = '#576a81'
midgreen = '#86a3c8'
lightgreen = '#a2c7f6'
colors = [lightgreen, midgreen, darkgreen, midgreen, lightgreen]
neg_colors = ["#f7318e", "#b02365", "#66143a", "#b02365", "#b02365"]
compartments = ["Exon (p.c.)", "ct", "Exon (n.c.)", "CpG islands"]
compartments = ["Exon (p.c.)", "ct", "Exon (n.c.)", "CpG islands"]

ngroups = len(compartments)
fig, axs = plt.subplots(nrows=ngroups // 2, ncols=2, figsize=(9, 7))
axs = axs.flatten() # needed to access each individual axis
bandwidth = 2
total_comparisons = len(set(methylation_g4_HG002meth_df["group"]))
print(f"Correcting for total comparisons: `{total_comparisons}`.")


meth_db = {"HG002": methylation_g4_HG002meth_df, "CHM13": methylation_g4_CHM13meth_df}

for tissue, group_df in meth_db.items():
    for i, compartment in enumerate(compartments):
        # subset the data for each word
        # temp = methylation_g4_df.filter(pl.col("comp") == compartment)
        temp = group_df.filter(pl.col("group") == compartment)

        print(compartment, temp["Conservation"].value_counts())
        cat = "avg_methylation"
        
        kde_subset = sns.kdeplot(
            data=temp,
            # hue="group",
            x=cat,
            fill=False,
            hue="Conservation",
            common_norm=False,
            palette={
                    "Conserved": midgreen,
                    "Non-conserved": "#f7318e"
                    },
            levels=100,
            lw=2.0,
            # color=sns.color_palette("Set3")[3],
            # palette={"Control": "green", 
            #          "Consensus Motif": "crimson"
            #         },
            bw_adjust = bandwidth,
            ax=axs[i],
            # color='grey',
            # edgecolor='black'
        )
        # if not kde_subset.lines:
        #    continue
        # sns.kdeplot(x=data, fill=True, color="red", ax=axs[i][0]) # temporarily draw a curve
        # x_values, y_values = axs[i][0].lines[0].get_data() # get the coordinates of the curve
        # axs[i][0].lines[0].remove()
        
        # mean value as a reference
        mean_pos = temp.filter(pl.col("Conservation") == "Conserved")[cat].mean()
        median_motif_pos = temp.filter(pl.col("Conservation") == "Conserved")[cat].median()
        
        mean_neg = temp.filter(pl.col("Conservation") == "Non-conserved")[cat].mean()
        median_motif_neg = temp.filter(pl.col("Conservation") == "Non-conserved")[cat].median()
        # median_control = subset.query("group == 'Control'")[cat].median()
        
        max_ = temp[cat].max()
        # print(f"Median Control: {median_control} \t Median G4: {median_motif}.")
        ylim = axs[i].get_ylim()[1]
        axs[i].plot([median_motif_pos, median_motif_pos], [0, ylim], linestyle='--', color=lightgreen)
        axs[i].plot([median_motif_neg, median_motif_neg], [0, ylim], linestyle='--', color="#f7318e")

        
        positive_ticks = [tick for tick in axs[i].get_yticks() if tick >= 0]
        axs[i].set_yticks(positive_ticks)

        if compartment == "enhancer" or compartment == "silencer":
            compartment = compartment.capitalize()
            
        if compartment.startswith("Protein Coding"):
            compartment_title = compartment.split(" ")[-1] + " (Protein Coding)"
        elif compartment.startswith("Non Coding"):
            compartment_title = compartment.split(" ")[-1] + " (Non Coding)"
        else:
            compartment_title = compartment
            
        axs[i].set_title(compartment_title)
        axs[i].title.set_size(18)
        # axs[i][j].axvline(0.2, lw=1.0, color='crimson')
        # axs[i][j].axvline(0.7, lw=1.0, color='crimson')
        axs[i].set_ylabel("Density", labelpad=20)
        
        axs[i].yaxis.label.set_size(18)
        axs[i].grid(lw=0.4, alpha=1.0)

        # axs[i].spines['top'].set_visible(False)
        # axs[i].spines['right'].set_visible(False)
        # axs[i].spines['left'].set_visible(False)
        axs[i].set_xlabel('')
        if i%2 == 1:
            axs[i].set_ylabel('')
        axs[i].legend(handles=[], frameon=False)
        axs[i].grid()
        axs[i].set_axisbelow(True)
        # axs[i].legend(loc=0, bbox_to_anchor=(1.01, 0.7))

        # compute quantiles
        arrays = [temp.filter(pl.col("Conservation") == "Conserved"), 
                temp.filter(pl.col("Conservation") == "Non-conserved")]
        axs[i].axhline(0.0, color='gray', lw=1.0)
        axs[i].axvline(0.5, color='gray', lw=1.0, zorder=0)
        axs[i].tick_params(axis="y", labelsize=13)
        axs[i].tick_params(axis="x", labelsize=16)

        
        # max_ = [subset_max, global_max]
        stat_tests = dict()
        group1 = arrays[0][cat]
        group2 = arrays[1][cat]
        
        stat, pval_unadjusted = ks_2samp(group1, group2, alternative="two-sided")
        stat, pval_unadjusted = mannwhitneyu(group1, group2, alternative="two-sided")
        effect = (2 * stat) / (len(group1) * len(group2)) - 1
        pval = pval_unadjusted * total_comparisons
        stat_tests[compartment] = (stat, pval, arrays)
        print(compartment, stat, pval, pval_unadjusted, effect)
                
        for array_id, array in enumerate(arrays):
            if array_id == 0:
                c = colors
            else:
                c = neg_colors
            print(array.shape)
            quantiles = np.percentile(array[cat], [2.5, 10, 25, 75, 90, 97.5])
            quantiles = quantiles.tolist()
            
            if pval < 0.001:
                stars = '***'
            elif pval < 0.01:
                stars = '**'
            elif pval < 0.05:
                stars = '*'
            else:
                stars = 'ns'
            
            if compartment == "SVA":
                lim = 1.3
                incremental = 0.2
                offset = 0.1
                offset_y = 0
            elif compartment == "Gene":
                lim = 0.4
                incremental = 0.083
                offset = 0.1
                offset_y = - 0.02
            elif compartment == "Exon (p.c.)":
                lim = 0.4
                incremental = 0.14
                offset = 0.12
                offset_y = - 0.1
            elif compartment == "CpG islands":
                lim = 0.9
                incremental = 0.3
                offset = 0.1
                offset_y = - 0.1
            elif compartment == "Exon (n.c.)":
                lim = 0.4
                incremental = 0.13
                offset = 0.12
                offset_y = - 0.01
            elif compartment == "CDS":
                lim = 0.3
                incremental = 0.069
                offset = 0.1
                offset_y = 0

            elif compartment == "3' UTR" or compartment == "5' UTR":
                lim = 0.6
                incremental = 0.17
                offset = 0.1
                offset_y = - 0.1
            elif compartment == "Silencer":
                lim = 1.0
                incremental = 0.1
                offset = 0.09
                offset_y = - 0.1
            elif compartment == "Enhancer":
                lim = 0.4
                incremental = 0.1
                offset = 0.1
                offset_y = - 0.05

            elif compartment == 'censat':
                incremental = 0.10
                lim = 0.4
                offset = 0.11
                offset_y = - 0.03

            elif compartment == "ct":
                incremental = 0.1
                lim = 0.3
                offset = 0.1
                offset_y = - 0.02

            else:
                lim = 0.4
                incremental = 0.12
                offset = 0.1
                offset_y = 0


        
            if array_id == 0:
                x = -0.1
                y = -lim
            else:
                x = -lim - 0.1
                y = -lim*2 
                
            for j in range(len(quantiles) - 1):
                axs[i].fill_between(
                [quantiles[j], quantiles[j+1]],
                x,
                y,
                color=c[j]
                )
                    

        axs[i].plot(
            [quantiles[j+1] + 0.05, quantiles[j+1] + 0.05],
            [(-0.1-lim)/2, (-lim-0.1-lim*2)/2],
            color='black',
            lw=1.0
        )
        if len(stars) == 1:
            stars = [' ', ' ', stars[0]]
            
        for star_id, star in enumerate(stars):
            increase = -incremental * star_id - 0.02
            axs[i].text(
            quantiles[j+1] + offset,
            increase + ((-0.1-lim)/2 + (-lim-0.1-lim*2)/2)/2 + offset_y,
            star,
            ha='center',
            va='bottom',
            fontsize=13,
            color='black',
        )
    
    fig.subplots_adjust(hspace=0.3)
    # plt.tight_layout()
    figures = Path(f"{os.getenv('SCRATCH')}/figures_g4_t2t/methylation")
    fig.savefig(f"{target_fig}/pangenome_G4_figures_methylation_group1_{tissue}.png", 
                transparent=True, 
                format="png", 
                dpi=300, 
                bbox_inches='tight')
            

In [ ]:
methylation_g4_CHM13meth_df

In [ ]:
temp

In [ ]:
from scipy.stats import mannwhitneyu, ks_2samp, gaussian_kde
from statsmodels.stats.proportion import proportions_ztest
import numpy as np
import seaborn as sns

HYPO_THRESHOLD = 0.2

def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

darkgreen  = '#576a81'
midgreen   = '#86a3c8'
lightgreen = '#a2c7f6'
palette_cons = {"Conserved": "#86a3c8", "Non-conserved": "#ff1cb3"}

colors     = [lightgreen, midgreen, darkgreen, midgreen, lightgreen]
neg_colors = [palette_cons["Non-conserved"], "#cc1690", "#991075", "#cc1690", palette_cons["Non-conserved"]]

compartments = ["Exon (p.c.)", "ct", "Exon (n.c.)", "CpG islands"]

ngroups = len(compartments)
bandwidth = 2
total_comparisons = len(set(methylation_g4_HG002meth_df["group"]))
print(f"Correcting for total comparisons: `{total_comparisons}`.")

lim_map = {
    "SVA":         (1.3, 0.2,   0.1,  0.0),
    "Gene":        (0.4, 0.083, 0.1,  -0.02),
    "Exon (p.c.)": (0.4, 0.14,  0.12, -0.00),
    "CpG islands": (0.9, 0.3,   0.1,  -0.1),
    "Exon (n.c.)": (0.4, 0.13,  0.12, +0.16),
    "CDS":         (0.3, 0.069, 0.1,  0.0),
    "3' UTR":      (0.6, 0.17,  0.1,  -0.1),
    "5' UTR":      (0.6, 0.17,  0.1,  -0.1),
    "Silencer":    (1.0, 0.1,   0.09, -0.1),
    "Enhancer":    (0.4, 0.1,   0.1,  -0.05),
    "censat":      (0.4, 0.10,  0.11, -0.03),
    "ct":          (0.3, 0.1,   0.1,  -0.02),
}


methylation_dfs = {"HG002": methylation_g4_HG002meth_df, 
                   "CHM13": methylation_g4_CHM13meth_df}

for db, group_df in methylation_dfs.items():
    fig, axs = plt.subplots(nrows=ngroups // 2, ncols=2, figsize=(10, 7))
    axs = axs.flatten()
    for i, compartment in enumerate(compartments):
        temp = group_df.filter(pl.col("group") == compartment)
        print(compartment, temp["Conservation"].value_counts())
        cat = "avg_methylation"

        sns.kdeplot(
            data=temp, x=cat, fill=False, hue="Conservation", common_norm=False,
            palette={"Conserved": palette_cons["Conserved"], "Non-conserved": palette_cons["Non-conserved"]},
            levels=100, lw=2.0, bw_adjust=bandwidth, ax=axs[i],
        )

        group1 = temp.filter(pl.col("Conservation") == "Conserved")[cat].drop_nulls().to_numpy()
        group2 = temp.filter(pl.col("Conservation") == "Non-conserved")[cat].drop_nulls().to_numpy()

        median_motif_pos = float(np.median(group1))
        median_motif_neg = float(np.median(group2))

        bw_fn = lambda x: x.scotts_factor() * bandwidth
        y_cons    = gaussian_kde(group1, bw_method=bw_fn)(median_motif_pos)[0]
        y_noncons = gaussian_kde(group2, bw_method=bw_fn)(median_motif_neg)[0]

        axs[i].plot([median_motif_pos, median_motif_pos], [0, y_cons],
                    linestyle='--', color=palette_cons["Conserved"])
        axs[i].plot([median_motif_neg, median_motif_neg], [0, y_noncons],
                    linestyle='--', color=palette_cons["Non-conserved"])

        positive_ticks = [tick for tick in axs[i].get_yticks() if tick >= 0]
        axs[i].set_yticks(positive_ticks)

        if compartment == "enhancer" or compartment == "silencer":
            compartment = compartment.capitalize()
        if compartment.startswith("Protein Coding"):
            compartment_title = compartment.split(" ")[-1] + " (Protein Coding)"
        elif compartment.startswith("Non Coding"):
            compartment_title = compartment.split(" ")[-1] + " (Non Coding)"
        else:
            compartment_title = compartment

        axs[i].set_title(compartment_title)
        axs[i].title.set_size(18)
        axs[i].set_ylabel("Density", labelpad=20)
        axs[i].yaxis.label.set_size(18)
        axs[i].yaxis.set_label_coords(-0.12, 0.65)
        axs[i].set_xlabel("")
        axs[i].set_xlim(axs[i].get_xlim()[0] * 0.7)
        if i % 2 == 1:
            axs[i].set_ylabel("")
        axs[i].legend(handles=[], frameon=False)
        axs[i].grid(lw=0.4, alpha=1.0)
        axs[i].set_axisbelow(True)
        axs[i].axhline(0.0, color="gray", lw=1.0)
        axs[i].axvline(0.5, color="gray", lw=1.0, zorder=0)
        axs[i].tick_params(axis="y", labelsize=13)
        axs[i].tick_params(axis="x", labelsize=16)

        arrays = [
            temp.filter(pl.col("Conservation") == "Conserved"),
            temp.filter(pl.col("Conservation") == "Non-conserved"),
        ]

        # proportions z-test on fraction hypomethylated
        k1, n1 = (group1 < HYPO_THRESHOLD).sum(), len(group1)
        k2, n2 = (group2 < HYPO_THRESHOLD).sum(), len(group2)
        frac_cons    = k1 / n1
        frac_noncons = k2 / n2
        _, pval_unadjusted = proportions_ztest([k1, k2], [n1, n2])
        h    = cohens_h(frac_cons, frac_noncons)
        pval = min(pval_unadjusted * total_comparisons, 1.0)
        print(f"{compartment}  frac_hypo_cons={frac_cons:.3f}  frac_hypo_noncons={frac_noncons:.3f}"
            f"  p_bonf={pval:.2e}  Cohen's h={h:.3f}")

        if pval < 0.001:   stars = "***"
        elif pval < 0.01:  stars = "**"
        elif pval < 0.05:  stars = "*"
        else:              stars = "ns"

        # pie insets: left=conserved, right=non-conserved
        for frac, color, x_pos in [
            (frac_cons,    palette_cons["Conserved"],     0.3),
            (frac_noncons, palette_cons["Non-conserved"], 0.5),
        ]:
            ax_pie = axs[i].inset_axes([x_pos, 0.7, 0.13, 0.38])
            ax_pie.pie(
                [frac, 1 - frac], colors=[color, "#e0e0e0"],
                startangle=90, wedgeprops=dict(linewidth=1.2, edgecolor="black"),
            )

        # quantile bands
        lim, incremental, offset, offset_y = lim_map.get(compartment, (0.4, 0.12, 0.1, 0.0))

        for array_id, array in enumerate(arrays):
            c = colors if array_id == 0 else neg_colors
            print(array.shape)
            quantiles = np.percentile(array[cat], [2.5, 10, 25, 75, 90, 97.5]).tolist()
            x = -0.1 if array_id == 0 else -lim - 0.1
            y = -lim  if array_id == 0 else -lim * 2
            for j in range(len(quantiles) - 1):
                axs[i].fill_between([quantiles[j], quantiles[j + 1]], x, y, color=c[j])

        axs[i].plot(
            [quantiles[j + 1] + 0.05, quantiles[j + 1] + 0.05],
            [(-0.1 - lim) / 2, (-lim - 0.1 - lim * 2) / 2],
            color="black", lw=1.0,
        )
        stars_list = [" ", " ", stars[0]] if len(stars) == 1 else list(stars)
        for star_id, star in enumerate(stars_list):
            increase = -incremental * star_id - 0.02
            axs[i].text(
                quantiles[j + 1] + offset,
                increase + ((-0.1 - lim) / 2 + (-lim - 0.1 - lim * 2) / 2) / 2 + offset_y,
                star, ha="center", va="bottom", fontsize=13, color="black",
            )
        axs[i].set_xlim(axs[i].get_xlim()[0] * 0.9, axs[i].get_xlim()[1] * 0.9)
        if i % 2 == 0:
            axs[i].set_ylabel("")
            axs[i].text(-0.22, 0.7, "Density", transform=axs[i].transAxes,
                        fontsize=18, rotation=90, va="center", ha="center")

    fig.subplots_adjust(hspace=0.3)
    fig.savefig(f"{target_fig}/pangenome_G4_figures_methylation_group1_{db}.png",
                transparent=True, format="png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{target_fig}/pangenome_G4_figures_methylation_group1_{db}.pdf",
                transparent=True, format="pdf", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
from scipy.stats import mannwhitneyu, ks_2samp, gaussian_kde
from statsmodels.stats.proportion import proportions_ztest
import numpy as np
import seaborn as sns

HYPO_THRESHOLD = 0.2

def cohens_h(p1, p2):
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

darkgreen  = '#576a81'
midgreen   = '#86a3c8'
lightgreen = '#a2c7f6'
palette_cons = {"Conserved": "#86a3c8", "Non-conserved": "#ff1cb3"}

colors     = [lightgreen, midgreen, darkgreen, midgreen, lightgreen]
neg_colors = [palette_cons["Non-conserved"], "#cc1690", "#991075", "#cc1690", palette_cons["Non-conserved"]]

compartments = ["Promoter (p.c.)", "Silencer", "Promoter (n.c.)", "Enhancer"]

ngroups = len(compartments)

bandwidth = 2
total_comparisons = len(set(methylation_g4_HG002meth_df["group"]))
print(f"Correcting for total comparisons: `{total_comparisons}`.")

lim_map = {
    "SVA":              (1.3, 0.2,   0.1,  0.0),
    "Protein Coding Genes": (0.4, 0.083, 0.1, 0.0),
    "Protein Coding Exons": (0.4, 0.085, 0.1, 0.0),
    "CDS":              (0.3, 0.08,  0.13, -0.05),
    "Promoter (p.c.)":  (0.6, 0.26,  0.12, -0.09),
    "Promoter (n.c.)":  (0.4, 0.13,  0.12, -0.05),
    "Silencer":         (1.0, 0.35,   0.2, -0.00),
    "Enhancer":         (0.35, 0.12,  0.1, -0.03),
    "censat":           (0.4, 0.07,  0.11,  0.0),
    "ct":               (0.4, 0.073, 0.1,   0.0),
}

for tissue, meth_db in methylation_dfs.items():
    fig, axs = plt.subplots(nrows=ngroups // 2, ncols=2, figsize=(10, 7))
    axs = axs.flatten()
    for i, compartment in enumerate(compartments):
        temp = meth_db.filter(pl.col("group") == compartment)
        print(compartment, temp["Conservation"].value_counts())
        cat = "avg_methylation"

        sns.kdeplot(
            data=temp, x=cat, fill=False, hue="Conservation", common_norm=False,
            palette={"Conserved": palette_cons["Conserved"], "Non-conserved": palette_cons["Non-conserved"]},
            levels=100, lw=2.0, bw_adjust=bandwidth, ax=axs[i],
        )

        group1 = temp.filter(pl.col("Conservation") == "Conserved")[cat].drop_nulls().to_numpy()
        group2 = temp.filter(pl.col("Conservation") == "Non-conserved")[cat].drop_nulls().to_numpy()

        median_motif_pos = float(np.median(group1))
        median_motif_neg = float(np.median(group2))

        bw_fn = lambda x: x.scotts_factor() * bandwidth
        y_cons    = gaussian_kde(group1, bw_method=bw_fn)(median_motif_pos)[0]
        y_noncons = gaussian_kde(group2, bw_method=bw_fn)(median_motif_neg)[0]

        axs[i].plot([median_motif_pos, median_motif_pos], [0, y_cons],
                    linestyle='--', color=palette_cons["Conserved"])
        axs[i].plot([median_motif_neg, median_motif_neg], [0, y_noncons],
                    linestyle='--', color=palette_cons["Non-conserved"])

        positive_ticks = [tick for tick in axs[i].get_yticks() if tick >= 0]
        axs[i].set_yticks(positive_ticks)

        if compartment == "enhancer" or compartment == "silencer":
            compartment = compartment.capitalize()
        if compartment.startswith("Protein Coding"):
            compartment_title = compartment.split(" ")[-1] + " (Protein Coding)"
        elif compartment.startswith("Non Coding"):
            compartment_title = compartment.split(" ")[-1] + " (Non Coding)"
        else:
            compartment_title = compartment

        axs[i].set_title(compartment_title)
        axs[i].title.set_size(18)
        axs[i].set_xlabel("")
        axs[i].set_xlim(axs[i].get_xlim()[0] * 0.7)
        if i % 2 == 1:
            axs[i].set_ylabel("")
        axs[i].legend(handles=[], frameon=False)
        axs[i].grid(lw=0.4, alpha=1.0)
        axs[i].set_axisbelow(True)
        axs[i].axhline(0.0, color="gray", lw=1.0)
        axs[i].axvline(0.5, color="gray", lw=1.0, zorder=0)
        axs[i].tick_params(axis="y", labelsize=13)
        axs[i].tick_params(axis="x", labelsize=16)

        arrays = [
            temp.filter(pl.col("Conservation") == "Conserved"),
            temp.filter(pl.col("Conservation") == "Non-conserved"),
        ]

        # proportions z-test on fraction hypomethylated
        k1, n1 = (group1 < HYPO_THRESHOLD).sum(), len(group1)
        k2, n2 = (group2 < HYPO_THRESHOLD).sum(), len(group2)
        frac_cons    = k1 / n1
        frac_noncons = k2 / n2
        _, pval_unadjusted = proportions_ztest([k1, k2], [n1, n2])
        h    = cohens_h(frac_cons, frac_noncons)
        pval = min(pval_unadjusted * total_comparisons, 1.0)
        print(f"{compartment}  frac_hypo_cons={frac_cons:.3f}  frac_hypo_noncons={frac_noncons:.3f}"
            f"  p_bonf={pval:.2e}  Cohen's h={h:.3f}")

        if pval < 0.001:   stars = "***"
        elif pval < 0.01:  stars = "**"
        elif pval < 0.05:  stars = "*"
        else:              stars = "ns"

        # pie insets: left=conserved, right=non-conserved
        for frac, color, x_pos in [
            (frac_cons,    palette_cons["Conserved"],     0.3),
            (frac_noncons, palette_cons["Non-conserved"], 0.5),
        ]:
            ax_pie = axs[i].inset_axes([x_pos, 0.7, 0.13, 0.38])
            ax_pie.pie(
                [frac, 1 - frac], colors=[color, "#e0e0e0"],
                startangle=90, wedgeprops=dict(linewidth=1.2, edgecolor="black"),
            )

        # quantile bands
        lim, incremental, offset, offset_y = lim_map.get(compartment, (0.4, 0.12, 0.1, 0.0))

        for array_id, array in enumerate(arrays):
            c = colors if array_id == 0 else neg_colors
            print(array.shape)
            quantiles = np.percentile(array[cat], [2.5, 10, 25, 75, 90, 97.5]).tolist()
            x = -0.1 if array_id == 0 else -lim - 0.1
            y = -lim  if array_id == 0 else -lim * 2
            for j in range(len(quantiles) - 1):
                axs[i].fill_between([quantiles[j], quantiles[j + 1]], x, y, color=c[j])

        axs[i].plot(
            [quantiles[j + 1] + 0.05, quantiles[j + 1] + 0.05],
            [(-0.1 - lim) / 2, (-lim - 0.1 - lim * 2) / 2],
            color="black", lw=1.0,
        )
        if len(stars) == 1:
            stars = [" ", " ", stars[0]]

        for star_id, star in enumerate(stars):
            if "n" in star:
                star = "   ns"
                offset_y = -0.2
                offset = 0.09
            increase = -incremental * star_id - 0.02
            axs[i].text(
                quantiles[j + 1] + offset,
                increase + ((-0.1 - lim) / 2 + (-lim - 0.1 - lim * 2) / 2) / 2 + offset_y,
                star, ha="center", va="bottom", fontsize=13, color="black",
            )
            if "n" in star:
                break

        axs[i].set_xlim(axs[i].get_xlim()[0] * 0.9, axs[i].get_xlim()[1] * 0.9)
        if i % 2 == 0:
            axs[i].set_ylabel("")
            axs[i].text(-0.22, 0.7, "Density", transform=axs[i].transAxes,
                        fontsize=18, rotation=90, va="center", ha="center")

    fig.subplots_adjust(hspace=0.3)
    fig.savefig(f"{target_fig}/pangenome_G4_figures_methylation_group2_{tissue}.png",
                transparent=True, format="png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{target_fig}/pangenome_G4_figures_methylation_group2_{tissue}.pdf",
                transparent=True, format="pdf", dpi=300, bbox_inches="tight")
    plt.show()


## Flanking Methylation

In [ ]:
import os
import numpy as np
import pyranges as pr
import matplotlib.pyplot as plt
from pathlib import Path

MAX_DIST = 2000
BIN_SIZE = 50
ALL_BINS = np.arange(-MAX_DIST, MAX_DIST, BIN_SIZE)

eg4_df = datasets["eG4"]

In [ ]:
regions_df = pd.read_table(Path(os.getenv('WORK')).joinpath("compartments_coords.tsv.gz"))
regions_bed = BedTool.from_dataframe(regions_df).sort()
regions_df

In [ ]:
comp_pr = pr.PyRanges(
    regions_df[["seqID", "start", "end", "group"]]
    .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
)
comp_pr

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyranges as pr
from pybedtools import BedTool

MAX_DIST = 2000
BIN_SIZE = 50
FLANK_BINS = np.arange(-MAX_DIST, 0, BIN_SIZE).tolist() + [0] + np.arange(BIN_SIZE, MAX_DIST + BIN_SIZE, BIN_SIZE).tolist()
FS = 20

eg4_in_comp_df = (
    pr.PyRanges(
        eg4_df[["seqID", "start", "end"]].drop_duplicates()
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
    ).join(comp_pr, suffix="_comp").df
    .pipe(lambda d: d[(d["Start"] >= d["Start_comp"]) & (d["End"] <= d["End_comp"])])
    [["Chromosome", "Start", "End", "group"]].drop_duplicates()
)

wins = eg4_in_comp_df.copy()
wins["g4_Start"] = wins["Start"]
wins["g4_End"]   = wins["End"]
wins["Start"]    = (wins["Start"] - MAX_DIST).clip(lower=0)
wins["End"]      = wins["End"] + MAX_DIST

meth_bed = BedTool.from_dataframe(methylation_HG002_df.iloc[:, :4]).sort()
itx_cols = ["win_chr", "win_start", "win_end", "g4_Start", "g4_End", "group",
            "meth_chr", "meth_start", "meth_end", "meth_level", "overlap"]

df_all = pd.read_csv(
    BedTool.from_dataframe(
        wins[["Chromosome", "Start", "End", "g4_Start", "g4_End", "group"]]
    ).sort().intersect(meth_bed, wo=True).fn,
    sep="\t", header=None, names=itx_cols,
)

# boundary distance: negative=upstream, 0=inside G4, positive=downstream
df_all["dist_bnd"] = np.where(
    df_all["meth_start"] < df_all["g4_Start"],
    df_all["meth_start"] - df_all["g4_Start"],
    np.where(
        df_all["meth_start"] > df_all["g4_End"],
        df_all["meth_start"] - df_all["g4_End"],
        0,
    )
)
df_all["bin"] = np.where(
    df_all["dist_bnd"] == 0,
    0,
    (df_all["dist_bnd"] // BIN_SIZE) * BIN_SIZE,
)
df_all = df_all[df_all["bin"].isin(FLANK_BINS)]

# min 1000 eG4s per compartment
comp_sizes = (df_all.groupby("group")
              .apply(lambda x: x[["g4_Start", "g4_End"]].drop_duplicates().shape[0]))
valid_comps = comp_sizes[comp_sizes >= 1000].index.tolist()

compartments = sorted(valid_comps)
heatmap_mat  = np.array([
    df_all[df_all["group"] == c].groupby("bin")["meth_level"]
    .median().reindex(FLANK_BINS).values
    for c in compartments
])
df_heat = pd.DataFrame(heatmap_mat, index=compartments, columns=FLANK_BINS)
df_heat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyranges as pr
from pybedtools import BedTool

MAX_DIST = 2000
BIN_SIZE = 50
FS = 20

upstream_bins   = np.arange(-MAX_DIST, 0, BIN_SIZE).tolist()        # -2000 .. -50
downstream_bins = np.arange(BIN_SIZE, MAX_DIST, BIN_SIZE).tolist()  # 50 .. 1950
FLANK_BINS      = upstream_bins + [0] + downstream_bins

eg4_in_comp_df = (
    pr.PyRanges(
        eg4_df[["seqID", "start", "end"]].drop_duplicates()
        .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
    ).join(comp_pr, suffix="_comp").df
    .pipe(lambda d: d[(d["Start"] >= d["Start_comp"]) & (d["End"] <= d["End_comp"])])
    [["Chromosome", "Start", "End", "group"]].drop_duplicates()
)

wins = eg4_in_comp_df.copy()
wins["g4_Start"] = wins["Start"]
wins["g4_End"]   = wins["End"]
wins["Start"]    = (wins["Start"] - MAX_DIST).clip(lower=0)
wins["End"]      = wins["End"] + MAX_DIST

meth_bed = BedTool.from_dataframe(methylation_HG002_df.iloc[:, :4]).sort()
itx_cols = ["win_chr", "win_start", "win_end", "g4_Start", "g4_End", "group",
            "meth_chr", "meth_start", "meth_end", "meth_level", "overlap"]

df_all = pd.read_csv(
    BedTool.from_dataframe(
        wins[["Chromosome", "Start", "End", "g4_Start", "g4_End", "group"]]
    ).sort().intersect(meth_bed, wo=True).fn,
    sep="\t", header=None, names=itx_cols,
)

# boundary distance
df_all["dist_bnd"] = np.where(
    df_all["meth_start"] < df_all["g4_Start"],
    df_all["meth_start"] - df_all["g4_Start"],   # negative (upstream)
    np.where(
        df_all["meth_start"] > df_all["g4_End"],
        df_all["meth_start"] - df_all["g4_End"],  # positive (downstream)
        0,                                         # inside G4
    )
)

# upstream: floor to bin boundary; downstream: ceiling to bin boundary; G4 body: 0
df_all["bin"] = np.where(
    df_all["dist_bnd"] == 0,
    0,
    np.where(
        df_all["dist_bnd"] < 0,
        (df_all["dist_bnd"] // BIN_SIZE) * BIN_SIZE,
        (np.ceil(df_all["dist_bnd"] / BIN_SIZE) * BIN_SIZE).astype(int),
    )
)
df_all = df_all[df_all["bin"].isin(FLANK_BINS)]

# min 1000 eG4s per compartment
comp_sizes  = (df_all.groupby("group")
               .apply(lambda x: x[["g4_Start", "g4_End"]].drop_duplicates().shape[0]))
valid_comps = comp_sizes[comp_sizes >= 1000].index.tolist()

compartments = sorted(valid_comps)
heatmap_mat  = np.array([
    df_all[df_all["group"] == c].groupby("bin")["meth_level"]
    .median().reindex(FLANK_BINS).values
    for c in compartments
])
df_heat = pd.DataFrame(heatmap_mat, index=compartments, columns=FLANK_BINS)

cg = sns.clustermap(
    df_heat,
    mask=df_heat.isna(),
    cmap="RdBu_r", vmin=0, vmax=1, center=0.5,
    col_cluster=False, row_cluster=True,
    figsize=(10, 12),
    cbar_pos=(0.92, 0.1, 0.03, 0.7),
    xticklabels=False, yticklabels=True,
    method="ward",
)
cg.ax_row_dendrogram.set_visible(False)
cg.ax_col_dendrogram.set_visible(False)

ax = cg.ax_heatmap
g4_col   = FLANK_BINS.index(0)
tick_vals = np.arange(-MAX_DIST, MAX_DIST, 500)
tick_vals = np.arange(-MAX_DIST, MAX_DIST + 500, 500)
tick_pos  = [FLANK_BINS.index(min(FLANK_BINS, key=lambda b: abs(b - v))) + 0.5 for v in tick_vals]

for y in range(1, len(df_heat)):
    ax.axhline(y, color="white", lw=2.0)

ax.axvline(g4_col, color="black", lw=2.5, ls="--", alpha=0.85)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_vals, fontsize=FS, rotation=45, ha="right")
ax.xaxis.set_tick_params(labelbottom=True)
ax.tick_params(axis="y", labelsize=FS + 2)
ax.set_xlabel("Distance from eG4 boundary (bp)", fontsize=FS + 4)
ax.yaxis.tick_left()
ax.yaxis.set_label_position("left")

cg.cax.tick_params(labelsize=FS)
cg.cax.set_ylabel("Median CpG Methylation", fontsize=FS + 2)

cg.fig.savefig(target_fig / "eg4_clustermap_per_compartment.png",
               dpi=300, 
               bbox_inches="tight", 
               transparent=True)
cg.fig.savefig(target_fig / "eg4_clustermap_per_compartment.pdf",
               dpi=300, 
               bbox_inches="tight", 
               transparent=True)
plt.show()

In [ ]:

cg = sns.clustermap(
    df_heat,
    mask=df_heat.isna(),
    cmap="RdBu_r", vmin=0, vmax=1, center=0.5,
    col_cluster=False, row_cluster=True,
    figsize=(10, 12),
    cbar_pos=(0.92, 0.1, 0.03, 0.7),
    xticklabels=False, yticklabels=True,
    method="ward",
)
cg.ax_row_dendrogram.set_visible(False)
cg.ax_col_dendrogram.set_visible(False)

ax = cg.ax_heatmap
g4_col   = FLANK_BINS.index(0)
tick_vals = np.arange(-MAX_DIST, MAX_DIST, 500)
tick_vals = np.arange(-MAX_DIST, MAX_DIST + 500, 500)
tick_pos  = [FLANK_BINS.index(min(FLANK_BINS, key=lambda b: abs(b - v))) + 0.5 for v in tick_vals]

for y in range(1, len(df_heat)):
    ax.axhline(y, color="white", lw=2.0)

ax.axvline(g4_col, color="black", lw=2.5, ls="--", alpha=0.85)
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_vals, fontsize=FS, rotation=45, ha="right")
ax.xaxis.set_tick_params(labelbottom=True)
ax.tick_params(axis="y", labelsize=FS + 2)
ax.set_xlabel("Distance from eG4 boundary (bp)", fontsize=FS + 4)
ax.yaxis.tick_left()
ax.yaxis.set_label_position("left")

cg.cax.tick_params(labelsize=FS)
cg.cax.set_ylabel("Median CpG Methylation", fontsize=FS + 2)

cg.fig.savefig(target_fig / "eg4_clustermap_per_compartment.png",
               dpi=300, 
               bbox_inches="tight", 
               transparent=True)
cg.fig.savefig(target_fig / "eg4_clustermap_per_compartment.pdf",
               dpi=300, 
               bbox_inches="tight", 
               transparent=True)
plt.show()

In [ ]:
for palette, fname in [
    ({"Conserved": "#86a3c8", "Non-conserved": "#ff1cb3"}, "legend_conservation.pdf"),
    ({"Background": "#a5acb0", "G4Hunter": "#ba8de0"},           "legend_g4_bg.pdf"),
]:
    handles = [mpatches.Patch(color=color, label=label) for label, color in palette.items()]
    fig, ax = plt.subplots(figsize=(1, 1.5))
    ax.set_axis_off()
    legend = ax.legend(
        handles=handles,
        fontsize=18,
        frameon=True, fancybox=True, shadow=True,
        loc="center",
    )
    fig.savefig(target_fig / fname,
                bbox_inches="tight", bbox_extra_artists=[legend],
                transparent=True)
    plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyranges as pr
from pybedtools import BedTool
from pathlib import Path

MAX_DIST    = 2000
BIN_SIZE    = 50
FLANK_CLASS = 20
HYPO_THRESH = 0.2

upstream_bins   = np.arange(-MAX_DIST, 0, BIN_SIZE).tolist()       # -2000 .. -50
downstream_bins = np.arange(BIN_SIZE, MAX_DIST, BIN_SIZE).tolist() # 50 .. 1950
FLANK_BINS      = upstream_bins + [0] + downstream_bins             # 80 bins total

eg4_pr = pr.PyRanges(
    eg4_df[["seqID", "start", "end"]].drop_duplicates()
    .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
)
eg4_in_comp_df = (
    eg4_pr.join(comp_pr, suffix="_comp").df
    .pipe(lambda d: d[(d["Start"] >= d["Start_comp"]) & (d["End"] <= d["End_comp"])])
    [["Chromosome", "Start", "End", "group"]].drop_duplicates()
)

eg4_in_comp_df["g4_Start"] = eg4_in_comp_df["Start"]
eg4_in_comp_df["g4_End"]   = eg4_in_comp_df["End"]
eg4_in_comp_df["Start"]    = (eg4_in_comp_df["Start"] - MAX_DIST).clip(lower=0)
eg4_in_comp_df["End"]      = eg4_in_comp_df["End"] + MAX_DIST

meth_bed = BedTool.from_dataframe(methylation_HG002_df.iloc[:, :4]).sort()
itx_cols = ["win_chr", "win_start", "win_end", "g4_Start", "g4_End", "group",
            "meth_chr", "meth_start", "meth_end", "meth_level", "overlap"]

df_all = pd.read_csv(
    BedTool.from_dataframe(
        eg4_in_comp_df[["Chromosome", "Start", "End", "g4_Start", "g4_End", "group"]]
    ).sort().intersect(meth_bed, wo=True).fn,
    sep="\t", header=None, names=itx_cols,
)

# boundary distance: negative = upstream of G4 start, 0 = inside G4, positive = downstream of G4 end
df_all["dist_bnd"] = np.where(
    df_all["meth_start"] < df_all["g4_Start"],
    df_all["meth_start"] - df_all["g4_Start"],
    np.where(
        df_all["meth_start"] > df_all["g4_End"],
        df_all["meth_start"] - df_all["g4_End"],
        0,
    ),
)
# upstream: floor division; downstream: ceiling division; body: 0
df_all["bin"] = np.where(
    df_all["dist_bnd"] == 0, 0,
    np.where(
        df_all["dist_bnd"] < 0,
        (df_all["dist_bnd"] // BIN_SIZE) * BIN_SIZE,
        (np.ceil(df_all["dist_bnd"] / BIN_SIZE) * BIN_SIZE).astype(int),
    ),
)
df_all = df_all[df_all["bin"].isin(FLANK_BINS)]
print(f"CpG observations: {len(df_all):,}")
print(f"Compartments: {sorted(df_all['group'].unique())}")


In [ ]:
comp_sizes = (
    df_all.groupby("group")
    .apply(lambda x: x[["win_chr", "g4_Start", "g4_End"]].drop_duplicates().shape[0])
)
valid_comps = comp_sizes[comp_sizes >= 1000].index.tolist()
print(f"Compartments kept (≥1000 eG4s): {valid_comps}")

heatmap_mat = np.array([
    df_all[df_all["group"] == c].groupby("bin")["meth_level"]
    .median().reindex(FLANK_BINS).values
    for c in sorted(valid_comps)
])
df_heat = pd.DataFrame(heatmap_mat, index=sorted(valid_comps), columns=FLANK_BINS)
df_heat


In [ ]:
target_fig

In [ ]:
FOLD_BINS = [0] + list(np.arange(BIN_SIZE, MAX_DIST, BIN_SIZE))

# tab20 handles any number of compartments
palette_comps = dict(zip(
    sorted(valid_comps_hypo),
    sns.color_palette("tab20", len(valid_comps_hypo))
))

ncols = 4
nrows = (len(valid_comps_hypo) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharey=True)
axes_flat = axes.flatten()

for i, comp in enumerate(sorted(valid_comps_hypo)):
    ax = axes_flat[i]
    sub = df_hypo[df_hypo["group"] == comp]
    medians, q25s, q75s = [], [], []
    for d in FOLD_BINS:
        vals = (sub[sub["dist_bnd"] == 0]["meth_level"] if d == 0
                else sub[sub["bin"].isin([-d, d])]["meth_level"])
        n_obs = len(vals)
        medians.append(vals.median()       if n_obs >= 5 else np.nan)
        q25s.append(vals.quantile(0.25)    if n_obs >= 5 else np.nan)
        q75s.append(vals.quantile(0.75)    if n_obs >= 5 else np.nan)

    c = palette_comps[comp]
    ax.plot(FOLD_BINS, medians, lw=2.5, color=c)
    ax.fill_between(FOLD_BINS, q25s, q75s, alpha=0.2, color=c)
    ax.axvline(0, color="black", lw=1.5, ls="--", alpha=0.6)
    ax.set_title(comp, fontsize=13)
    ax.set_xlim(0, MAX_DIST)
    ax.set_ylim(0, 1)
    ax.grid(lw=0.4, alpha=0.6)
    ax.tick_params(labelsize=11)
    if i % ncols == 0:
        ax.set_ylabel("Median CpG Methylation", fontsize=13)
    if i >= (nrows - 1) * ncols:
        ax.set_xlabel("Distance from eG4 boundary (bp)", fontsize=13)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.tight_layout()
fig.savefig(target_fig / "eg4_hypo_meth_rise_per_compartment.pdf",
            bbox_inches="tight", transparent=True)
plt.show()


## Ternary Plots

In [ ]:
regions_df = pd.read_table(Path(os.getenv('WORK')).joinpath("compartments_coords.tsv.gz"))
regions_bed = BedTool.from_dataframe(regions_df).sort()

In [ ]:
meth_annotated["G4Hunter", "HG002"]

In [ ]:
from tqdm import tqdm

meth_levels =  ["Hypomethylated", "Methylated", "Hypermethylated"]
methylation_g4_HG002meth_pyramid_df = meth_annotated["G4Hunter", "HG002"][["seqID", "start", "end", "methylation_level"]].copy()
g4_HG002_meth_bed = dict()
for level in tqdm(meth_levels):
    g4_HG002_meth_bed[level] = BedTool.from_dataframe(methylation_g4_HG002meth_pyramid_df[methylation_g4_HG002meth_pyramid_df["methylation_level"] == level]).sort()
    
g4_HG002_meth_bed

In [ ]:
import numpy as np 
from scipy.stats import beta, dirichlet
import matplotlib.pyplot as plt
import ternary
from scipy.stats import multinomial

COVERAGE_FIELDS = ["total_hits", "total_bases", "compartment_length", "coverage"]
def extract_g4_meth(regions_bed, g4_bed, meth_levels):
    g4_meth_compartment = None
    print(g4_bed.keys())
    for level in tqdm(meth_levels):
        temp = pl.read_csv(
                            regions_bed.coverage(g4_bed[level]).fn,
                            has_header=False,
                            separator="\t",
                            new_columns=["seqID", "start", "end", "compartment"] + COVERAGE_FIELDS
        ).drop(['compartment_length'])
        temp = temp.rename({"total_bases": f"total_bases_{level}",
                            "total_hits": f"total_hits_{level}",
                            "coverage": f"coverage_{level}"})
        if g4_meth_compartment is None:
            g4_meth_compartment = temp
        else:
            g4_meth_compartment = g4_meth_compartment.join(
                            temp,
                            on=["seqID", "start", "end", "compartment"],
                            suffix=f"_{level}"
            )
    return g4_meth_compartment

g4_meth_compartment_HG002 = extract_g4_meth(regions_bed, 
                                            g4_HG002_meth_bed, 
                                            meth_levels)
g4_meth_compartment_HG002

In [ ]:
g4_meth_compartment_grped = (
    g4_meth_compartment_HG002
    .group_by("compartment")
    .agg([pl.col(f"total_hits_{level}").sum() for level in meth_levels])
)

In [ ]:
target_concentration = 100
prior = np.array([1.0, 1.0, 1.0])  # Uniform prior for 3 categories
alpha_vectors = dict()
for row in g4_meth_compartment_grped.iter_rows(named=True):
    compartment = row['compartment']
    counts = np.array([
        row['total_hits_Hypomethylated'], 
        row['total_hits_Methylated'], 
        row['total_hits_Hypermethylated']
    ])
    total = counts.sum()
    if total == 0:
        continue
    
    posterior_alpha = prior + counts
    
    # Rescale to target concentration while preserving proportions
    # This is essentially what your original loop was doing!
    scaled_alpha = target_concentration * posterior_alpha / posterior_alpha.sum()
    
    print(compartment, scaled_alpha)
    alpha_vectors[compartment] = scaled_alpha

In [ ]:
import plotly.figure_factory as ff
import numpy as np
from scipy.special import gamma
import plotly.graph_objects as go

class Dirichlet(object):
    def __init__(self, alpha, compartment=None):
        self.compartment = compartment
        self._alpha = np.array(alpha)
        self._coef = gamma(np.sum(self._alpha)) / np.multiply.reduce([gamma(a) for a in self._alpha])
    
    def pdf(self, x):
        x = np.array(x)
        # Handle zeros to avoid log(0)
        if np.any(x <= 0):
            return 0.0
        return self._coef * np.multiply.reduce([xx ** (aa - 1) for (xx, aa) in zip(x, self._alpha)])
from scipy.special import gammaln

# class Dirichlet(object):
#     def __init__(self, alpha, compartment=None):
#         self.compartment = compartment
#         self._alpha = np.array(alpha)
#         self._log_coef = gammaln(np.sum(self._alpha)) - np.sum(gammaln(self._alpha))
    
#     def pdf(self, x):
#         x = np.array(x)
#         # Handle zeros to avoid log(0)
#         if np.any(x <= 0):
#             return 0.0
#         log_vals = np.sum([(aa - 1) * np.log(xx) for xx, aa in zip(x, self._alpha)])
#         return np.exp(self._log_coef + log_vals)

def generate_ternary_grid(n=50):
    Al = np.linspace(0, 1, n)
    Cu = np.linspace(0, 1, n)
    X, Y = np.meshgrid(Al, Cu)
    Z = 1 - X - Y
    mask = (X + Y <= 1)
    X = X[mask]
    Y = Y[mask]
    Z = Z[mask]
    return np.column_stack([X, Y, Z])
    
compartments = ['Silencer',
                'Enhancer',
                'rDNA', 
                'SVA',
                'censat', 
                'ct', 
                'Exon (n.c.)', 
                'Gene (n.c.)', 
                'Exon (p.c.)', 
                'Gene (p.c.)', 
                'Promoter (p.c.)', 
                'Promoter (n.c.)', 
                'Gene (n.c.)', 
                'CDS',
               ]

for compartment in compartments:
    print(compartment)
    alpha = alpha_vectors[compartment]
    print(alpha)
    dist = Dirichlet(alpha)
    points = generate_ternary_grid(n=50)
    pdf_values = np.array([dist.pdf(point) for point in points])
    X = points[:, 0]
    Y = points[:, 1]
    Z = points[:, 2]
    
    fig = ff.create_ternary_contour(np.array([X, Y, Z]), pdf_values,
                                    pole_labels=['', '', ''],
                                    interp_mode='cartesian',
                                    ncontours=20,
                                    colorscale='Jet',
                                    showscale=True,
                                    title=dict(text=''))
    
    corners = np.array([
        [1, 0, 0], 
        [0, 1, 0], 
        [0, 0, 1] 
    ])
    
    barycenter = corners.mean(axis=0)
    
    # Add dashed lines from barycenter to each corner
    for corner in corners:
        fig.add_trace(go.Scatterternary(
            a=[barycenter[0], corner[0]],
            b=[barycenter[1], corner[1]],
            c=[barycenter[2], corner[2]],
            mode='lines',
            line=dict(color='white', width=2, dash='dash'),
            showlegend=False
        ))
    
    # Add barycenter marker
    fig.add_trace(go.Scatterternary(
        a=[barycenter[0]],
        b=[barycenter[1]],
        c=[barycenter[2]],
        mode='markers+text',
        marker=dict(color='white', size=8, symbol='circle'),
        text=[''],
        textposition='top center',
        showlegend=False
    ))
    fig.update_layout(
        ternary=dict(
            aaxis=dict(title=dict(font=dict(size=26))),
            baxis=dict(title=dict(font=dict(size=26))),
            caxis=dict(title=dict(font=dict(size=26)))
        )
    )
    
    fig.update_traces(marker_colorbar_title='Probability Density', selector=dict(marker_showscale=True))
    fig.update_layout(width=1000, height=700)
    fig.update_traces(
            selector=dict(marker_showscale=True),
            marker_colorbar=dict(
                title=dict(
                    text='<br>Probability Density',  # manual spacing
                    font=dict(size=20),
                ),
                tickfont=dict(size=18), # change to desired size
                thickness=30
            )
    )
    
    fig.update_layout(
        ternary=dict(
            aaxis=dict(
                tickmode='linear',
                tick0=0,
                dtick=0.1,  # Tick every 0.1 (i.e., every 10%)
                ticks='outside',
                tickfont=dict(size=20)
            ),
            baxis=dict(
                tickmode='linear',
                tick0=0,
                dtick=0.1,
                ticks='outside',
                tickfont=dict(size=20)
            ),
            caxis=dict(
                tickmode='linear',
                tick0=0,
                dtick=0.1,
                ticks='outside',
                tickfont=dict(size=20)
            )
        )
    )
    fig.update_traces(
        selector=dict(marker_showscale=True),
        marker_colorbar=dict(
            title=dict(text='', font=dict(size=19))
        )
    )

    fig.show()
    fig.write_image(f"{target_fig}/ternary_plot_{compartment}.png", format="png", scale=300/96)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from scipy.special import gammaln
import os

# Ternary coordinate helpers
_c = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, 3**0.5 / 2]])
_AREA = 0.5 * 3**0.5 / 2
_pairs = [_c[np.roll(range(3), -i)[1:]] for i in range(3)]

def xy2bc(xy, tol=1e-4):
    def _area(xy, pair):
        v0, v1 = pair[0] - np.asarray(xy), pair[1] - np.asarray(xy)
        return 0.5 * abs(v0[0]*v1[1] - v0[1]*v1[0])
    return np.clip([_area(xy, p) / _AREA for p in _pairs], tol, 1 - tol)

class Dirichlet:
    def __init__(self, alpha, compartment=None):
        self.compartment = compartment
        self._alpha = np.array(alpha, dtype=float)
        self._log_coef = gammaln(self._alpha.sum()) - gammaln(self._alpha).sum()

    def pdf(self, x):
        x = np.asarray(x, dtype=float)
        if np.any(x <= 0):
            return 0.0
        return np.exp(self._log_coef + np.sum((self._alpha - 1) * np.log(x)))

def _add_ticks(ax, tick_interval=0.1, fontsize=12):
    tick_len = 0.025
    label_gap = 0.01
    centroid = _c.mean(axis=0)
    edge_cfg = [
        dict(rotation=0,   ha='center', va='top'),
        dict(rotation=-60, ha='left',   va='center'),
        dict(rotation=60,  ha='right',  va='center'),
    ]
    for i in range(3):
        c0, c1 = _c[i], _c[(i + 1) % 3]
        edge = c1 - c0
        perp = np.array([-edge[1], edge[0]])
        perp /= np.linalg.norm(perp)
        if np.dot(perp, centroid - c0) < 0:
            perp = -perp
        for t in np.arange(tick_interval, 1.0, tick_interval):
            p = c0 + t * edge
            ax.plot([p[0], p[0] + tick_len * perp[0]],
                    [p[1], p[1] + tick_len * perp[1]], 'k-', lw=1.0)
            lp = p - label_gap * perp
            ax.text(lp[0], lp[1], f'{t:.1f}', fontsize=fontsize, **edge_cfg[i])


def draw_pdf_contours(dist, corner_labels, nlevels=20, subdiv=8):
    fig, ax = plt.subplots(figsize=(8, 7))
    refiner = tri.UniformTriRefiner(tri.Triangulation(_c[:, 0], _c[:, 1]))
    trimesh = refiner.refine_triangulation(subdiv=subdiv)
    pvals = np.array([dist.pdf(xy2bc([x, y])) for x, y in zip(trimesh.x, trimesh.y)])

    contour = ax.tricontourf(trimesh, pvals, nlevels, cmap='jet')
    cbar = fig.colorbar(contour, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Probability Density', fontsize=18)
    cbar.ax.tick_params(labelsize=16)

    # Triangle outline + ticks
    outline = np.vstack([_c, _c[0]])
    ax.plot(outline[:, 0], outline[:, 1], 'k-', lw=1.5)
    _add_ticks(ax)

    # Barycenter + dashed lines to corners
    bc = _c.mean(axis=0)
    ax.scatter(*bc, color='white', s=80, zorder=4)
    for corner in _c:
        ax.plot([bc[0], corner[0]], [bc[1], corner[1]], color='white', ls='--', lw=1.5)

    # Corner labels
    offsets = [np.array([-0.08, -0.06]), np.array([0.08, -0.06]), np.array([0.0, 0.06])]
    ha_list = ['right', 'left', 'center']
    va_list = ['top', 'top', 'bottom']
    for i, (label, offset, ha, va) in enumerate(zip(corner_labels, offsets, ha_list, va_list)):
        pos = _c[i] + offset
        ax.text(pos[0], pos[1], label, fontsize=18, ha=ha, va=va)

    ax.set_aspect('equal')
    ax.axis('off')
    return fig, ax


compartments = [
    'Silencer', 'Enhancer', 'rDNA', 'SVA', 'censat', 'ct',
    'Exon (n.c.)', 'Gene (n.c.)', 'Exon (p.c.)', 'Gene (p.c.)',
    'Promoter (p.c.)', 'Promoter (n.c.)', 'Gene (n.c.)', 'CDS',
]

os.makedirs(target_fig, exist_ok=True)
for compartment in compartments:
    dist = Dirichlet(alpha_vectors[compartment], compartment)
    fig, ax = draw_pdf_contours(dist, corner_labels=meth_levels)
    fig.savefig(f"{target_fig}/ternary_plot_{compartment}.png", dpi=300, bbox_inches='tight', transparent=True)
    plt.show()
    plt.close(fig)
    print(f"Saved: {compartment}")

In [ ]:
target_fig